# Step 6 - Comparison and Explainability

Executed project evidence and reproducible code.

# Step 6 — Compare outputs by risk, expected return, turnover, guardrails, breaches, and explainability

In [46]:
from pathlib import Path
import importlib
import inspect
import json
import re
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import files
STEP6_REQUIRED_OBJECTS = ['step4', 'portfolio_data', 'current_weights', 'stages', 'constraints', 'trading_config', 'STEP5_DAILY_RETURNS', 'PRIMARY_CONTEXT', 'PRIMARY_SCENARIOS', 'STEP5_PROFILE_RESULTS', 'RISK_POLICY_MODE', 'DATA_SOURCE', 'STEP4_COST_SCENARIO', 'OUTPUT_ROOT']
STEP6_MISSING_OBJECTS = [name for name in STEP6_REQUIRED_OBJECTS if name not in globals()]
if STEP6_MISSING_OBJECTS:
    raise RuntimeError('Run Steps 3–5Q first. Missing Step 6 inputs: ' + ', '.join(STEP6_MISSING_OBJECTS))
print('PASS: Step 6 runtime contract is complete.')
print('Profiles received:', len(STEP5_PROFILE_RESULTS))
print('Selected risk policy:', RISK_POLICY_MODE)
print('Data source:', DATA_SOURCE)


PASS: Step 6 runtime contract is complete.
Profiles received: 7
Selected risk policy: soft_warning
Data source: synthetic


## Install the Step 6 comparison module inside the Colab runtime

In [47]:
%%writefile step_06_comparison_final.py
from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Mapping, MutableMapping, Sequence
import numpy as np
import pandas as pd
EPS = 1e-12

def _array(values: Any) -> np.ndarray:
    if hasattr(values, 'to_numpy'):
        return np.asarray(values.to_numpy(dtype=float), dtype=float)
    return np.asarray(values, dtype=float)

@dataclass(frozen=True)
class Step6Context:
    step4: Any
    portfolio_data: Any
    current_weights: Any
    scenarios: Any
    constraints: Any
    trading_config: Any
    daily_returns: pd.DataFrame
    hard_tolerance: float = 5e-06
    binding_tolerance: float = 1e-05
    duplicate_tolerance: float = 1e-08
    material_weight_threshold: float = 0.005
    material_trade_threshold: float = 0.001

    def validate(self) -> None:
        tickers = list(self.portfolio_data.tickers)
        n_assets = len(tickers)
        current = _array(self.current_weights)
        if current.shape != (n_assets,):
            raise ValueError('current_weights has the wrong shape.')
        if not np.isclose(current.sum(), 1.0, atol=1e-08):
            raise ValueError('current_weights must sum to one.')
        if list(self.daily_returns.columns) != tickers:
            raise ValueError('daily_returns columns must exactly match the portfolio ticker order.')
        if self.daily_returns.empty:
            raise ValueError('daily_returns must not be empty.')
        if not np.isfinite(self.daily_returns.to_numpy(dtype=float)).all():
            raise ValueError('daily_returns contains non-finite values.')
        self.portfolio_data.validate()
        self.constraints.validate(self.portfolio_data)
        self.scenarios.validate(n_assets)

def register_candidate(registry: MutableMapping[str, dict[str, Any]], *, context: Step6Context, label: str, family: str, role: str, method: str, weights: Any, source_name: str, decision_eligible: bool, independent_selection: bool, method_traceability: float, metadata: Mapping[str, Any] | None=None) -> None:
    if label in registry:
        raise ValueError(f'Candidate already registered: {label}')
    vector = _array(weights).reshape(-1)
    n_assets = len(context.portfolio_data.tickers)
    if vector.shape != (n_assets,):
        raise ValueError(f'{label}: weights have the wrong shape.')
    if not np.isfinite(vector).all():
        raise ValueError(f'{label}: weights contain non-finite values.')
    if abs(float(vector.sum()) - 1.0) > 1e-05:
        raise ValueError(f'{label}: weights do not sum to one.')
    if float(vector.min()) < -1e-06:
        raise ValueError(f'{label}: negative weight detected.')
    if not 0.0 <= float(method_traceability) <= 1.0:
        raise ValueError('method_traceability must lie in [0, 1].')
    registry[label] = {'label': label, 'family': family, 'role': role, 'method': method, 'weights': vector.copy(), 'source_name': source_name, 'decision_eligible': bool(decision_eligible), 'independent_selection': bool(independent_selection), 'method_traceability': float(method_traceability), 'metadata': dict(metadata or {})}

def _path_metrics(daily_returns: pd.DataFrame, weights: np.ndarray) -> dict[str, float]:
    path = daily_returns.to_numpy(dtype=float) @ weights
    path = np.maximum(path, -0.999999)
    wealth = np.cumprod(1.0 + path)
    running_peak = np.maximum.accumulate(wealth)
    drawdown = 1.0 - wealth / np.maximum(running_peak, EPS)
    n_days = len(path)
    annualized_return = float(wealth[-1] ** (252.0 / max(n_days, 1)) - 1.0)
    annualized_volatility = float(np.std(path, ddof=1) * np.sqrt(252.0))
    quantile_05 = float(np.quantile(path, 0.05))
    tail = path[path <= quantile_05 + EPS]
    daily_var_95 = max(-quantile_05, 0.0)
    daily_cvar_95 = max(-float(tail.mean()), 0.0) if len(tail) else daily_var_95
    return {'in_sample_annualized_return': annualized_return, 'in_sample_annualized_volatility': annualized_volatility, 'in_sample_maximum_drawdown': float(drawdown.max()), 'daily_var_95': daily_var_95, 'daily_cvar_95': daily_cvar_95}

def _policy_scale(value: float, lower: float, upper: float) -> float:
    finite = [abs(float(x)) for x in (value, lower, upper) if np.isfinite(x)]
    return max(finite + [0.01])

def _decorate_audit(audit: pd.DataFrame, *, tolerance: float, binding_tolerance: float) -> pd.DataFrame:
    frame = audit.copy()
    lower = frame['lower'].to_numpy(dtype=float)
    upper = frame['upper'].to_numpy(dtype=float)
    value = frame['value'].to_numpy(dtype=float)
    lower_violation = np.where(np.isfinite(lower), np.maximum(lower - value, 0.0), 0.0)
    upper_violation = np.where(np.isfinite(upper), np.maximum(value - upper, 0.0), 0.0)
    absolute_violation = np.maximum(lower_violation, upper_violation)
    scales = np.asarray([_policy_scale(v, lo, hi) for v, lo, hi in zip(value, lower, upper, strict=True)], dtype=float)
    frame['lower_violation'] = lower_violation
    frame['upper_violation'] = upper_violation
    frame['absolute_violation'] = absolute_violation
    frame['normalized_violation'] = absolute_violation / scales
    frame['satisfied'] = (lower_violation <= tolerance) & (upper_violation <= tolerance)
    zero_lower_bound = np.isclose(frame['lower'], 0.0)
    zero_realized_exposure = frame['value'].abs() <= binding_tolerance
    supported_nonnegativity_row = frame['category'].eq('asset') & frame['constraint'].str.startswith('weight_') | frame['category'].eq('asset_class') & frame['constraint'].str.startswith('class_')
    trivial_lower_zero = zero_lower_bound & zero_realized_exposure & supported_nonnegativity_row
    excluded_policy_rows = frame['category'].eq('budget') | frame['constraint'].eq('minimum_weight') | frame['constraint'].eq('trade_accounting_max_abs_error')
    frame['is_policy_guardrail'] = ~excluded_policy_rows
    frame['trivial_nonnegativity_bound'] = trivial_lower_zero
    lower_margin = np.where(np.isfinite(lower), (value - lower) / scales, np.inf)
    upper_margin = np.where(np.isfinite(upper), (upper - value) / scales, np.inf)
    lower_margin = np.where(trivial_lower_zero, np.inf, lower_margin)
    normalized_margin = np.minimum(lower_margin, upper_margin)
    normalized_margin = np.where(trivial_lower_zero, np.nan, normalized_margin)
    normalized_margin = np.where(frame['is_policy_guardrail'], normalized_margin, np.nan)
    frame['policy_normalized_margin'] = normalized_margin
    lower_active = np.isfinite(lower) & (np.abs(value - lower) <= binding_tolerance)
    upper_active = np.isfinite(upper) & (np.abs(upper - value) <= binding_tolerance)
    frame['mathematical_active_bound'] = lower_active | upper_active
    frame['binding_policy_guardrail'] = frame['is_policy_guardrail'] & frame['satisfied'] & ~frame['trivial_nonnegativity_bound'] & (frame['policy_normalized_margin'] <= binding_tolerance)
    return frame

def _scenario_audit(*, scenarios: Any, weights: np.ndarray, tolerance: float) -> pd.DataFrame:
    losses = _array(scenarios.loss_matrix) @ weights
    warnings = _array(scenarios.warning_thresholds)
    if hasattr(scenarios, 'hard_loss_limits'):
        hard_limits = _array(scenarios.hard_loss_limits)
    elif hasattr(scenarios, 'hard_limits'):
        hard_limits = _array(scenarios.hard_limits)
    else:
        raise AttributeError('Scenario set has no hard-limit field.')
    return pd.DataFrame({'scenario_loss': losses, 'warning_threshold': warnings, 'hard_limit': hard_limits, 'warning_headroom': warnings - losses, 'warning_excess': np.maximum(losses - warnings, 0.0), 'warning_satisfied': losses <= warnings + tolerance, 'hard_limit_headroom': hard_limits - losses, 'hard_limit_excess': np.maximum(losses - hard_limits, 0.0), 'hard_limit_satisfied': losses <= hard_limits + tolerance}, index=list(scenarios.names))

def _attribution_tables(*, context: Step6Context, weights: np.ndarray) -> dict[str, pd.DataFrame]:
    data = context.portfolio_data
    tickers = list(data.tickers)
    current = _array(context.current_weights)
    trade = weights - current
    covariance = _array(data.covariance)
    marginal_variance = covariance @ weights
    variance = float(weights @ marginal_variance)
    volatility = float(np.sqrt(max(variance, 0.0)))
    growth_contribution = weights * _array(data.growth)
    income_contribution = weights * _array(data.income)
    total_return_contribution = growth_contribution + income_contribution
    variance_contribution = weights * marginal_variance
    if volatility > EPS:
        volatility_contribution = variance_contribution / volatility
    else:
        volatility_contribution = np.zeros_like(weights)
    linear_cost_contribution = np.abs(trade) * _array(data.linear_cost)
    impact_vector = _array(data.impact_matrix) @ trade
    impact_cost_contribution = trade * impact_vector
    turnover_contribution = np.abs(trade)
    scenario_losses = _array(context.scenarios.loss_matrix) @ weights
    worst_index = int(np.argmax(scenario_losses))
    worst_scenario_vector = _array(context.scenarios.loss_matrix)[worst_index]
    worst_scenario_contribution = weights * worst_scenario_vector
    asset = pd.DataFrame({'asset_class': list(data.asset_classes), 'description': list(data.descriptions), 'weight': weights, 'current_weight': current, 'trade': trade, 'absolute_trade': np.abs(trade), 'growth_contribution': growth_contribution, 'income_contribution': income_contribution, 'expected_return_contribution': total_return_contribution, 'variance_contribution': variance_contribution, 'volatility_contribution': volatility_contribution, 'turnover_contribution': turnover_contribution, 'linear_cost_contribution': linear_cost_contribution, 'impact_cost_contribution': impact_cost_contribution, 'total_cost_contribution': linear_cost_contribution + impact_cost_contribution, 'worst_scenario_loss_contribution': worst_scenario_contribution}, index=tickers)
    numeric_columns = ['weight', 'current_weight', 'trade', 'absolute_trade', 'growth_contribution', 'income_contribution', 'expected_return_contribution', 'variance_contribution', 'volatility_contribution', 'turnover_contribution', 'linear_cost_contribution', 'impact_cost_contribution', 'total_cost_contribution', 'worst_scenario_loss_contribution']
    by_class = asset.groupby('asset_class')[numeric_columns].sum().sort_index()
    scenario_detail = pd.DataFrame(_array(context.scenarios.loss_matrix) * weights[None, :], index=list(context.scenarios.names), columns=tickers)
    return {'asset': asset, 'asset_class': by_class, 'scenario_asset': scenario_detail, 'worst_scenario_name': pd.DataFrame({'worst_scenario_name': [context.scenarios.names[worst_index]]})}

def _absolute_top_coverage(values: np.ndarray, count: int=5) -> float:
    absolute = np.abs(np.asarray(values, dtype=float))
    denominator = float(absolute.sum())
    if denominator <= EPS:
        return 1.0
    return float(np.sort(absolute)[::-1][:count].sum() / denominator)

def _evaluate_candidate(*, context: Step6Context, candidate: Mapping[str, Any]) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame, dict[str, pd.DataFrame]]:
    data = context.portfolio_data
    weights = _array(candidate['weights'])
    current = _array(context.current_weights)
    trade = weights - current
    buys = np.maximum(trade, 0.0)
    sells = np.maximum(-trade, 0.0)
    growth = float(_array(data.growth) @ weights)
    income = float(_array(data.income) @ weights)
    expected_return = growth + income
    variance = float(weights @ _array(data.covariance) @ weights)
    volatility = float(np.sqrt(max(variance, 0.0)))
    gross_turnover = float(np.abs(trade).sum())
    linear_cost = float(_array(data.linear_cost) @ np.abs(trade))
    impact_cost = float(trade @ _array(data.impact_matrix) @ trade)
    total_cost = linear_cost + impact_cost
    concentration = float(weights @ weights)
    effective_holdings = 1.0 / concentration if concentration > EPS else np.inf
    scenario = _scenario_audit(scenarios=context.scenarios, weights=weights, tolerance=context.hard_tolerance)
    raw_audit = context.step4.audit_constraints(data=data, weights=weights, current_weights=current, buys=buys, sells=sells, scenarios=context.scenarios, constraint_config=context.constraints, trading_config=context.trading_config, check_asset_caps=True, check_classes=True, check_factors=True, check_income=True, check_return=True, check_trading=True, check_scenario_hard=True, tolerance=context.hard_tolerance)
    audit = _decorate_audit(raw_audit, tolerance=context.hard_tolerance, binding_tolerance=context.binding_tolerance)
    attributions = _attribution_tables(context=context, weights=weights)
    asset_attr = attributions['asset']
    policy_rows = audit.loc[audit['is_policy_guardrail']]
    policy_margins = policy_rows['policy_normalized_margin'].replace([np.inf, -np.inf], np.nan).dropna()
    if policy_margins.empty:
        headroom_min = np.nan
        headroom_10 = np.nan
        headroom_median = np.nan
    else:
        headroom_min = float(policy_margins.min())
        headroom_10 = float(policy_margins.quantile(0.1))
        headroom_median = float(policy_margins.median())
    hard_breaches = audit.loc[~audit['satisfied']]
    warning_breaches = scenario.loc[~scenario['warning_satisfied']]
    path_metrics = _path_metrics(context.daily_returns, weights)
    top5_weight_share = float(np.sort(weights)[::-1][:5].sum())
    top10_weight_share = float(np.sort(weights)[::-1][:10].sum())
    material_holdings_count = int(np.count_nonzero(weights >= context.material_weight_threshold))
    active_trade_count = int(np.count_nonzero(np.abs(trade) >= context.material_trade_threshold))
    row: dict[str, Any] = {'candidate': candidate['label'], 'family': candidate['family'], 'role': candidate['role'], 'method': candidate['method'], 'source_name': candidate['source_name'], 'decision_eligible_declared': candidate['decision_eligible'], 'independent_selection': candidate['independent_selection'], 'method_traceability': candidate['method_traceability'], 'expected_growth': growth, 'income_yield': income, 'expected_total_return': expected_return, 'variance': variance, 'volatility': volatility, 'return_to_volatility': expected_return / volatility if volatility > EPS else np.nan, 'worst_scenario_loss': float(scenario['scenario_loss'].max()), 'weighted_scenario_loss': float(_array(context.scenarios.weights) @ scenario['scenario_loss'].to_numpy(dtype=float)), 'gross_turnover': gross_turnover, 'one_way_turnover': 0.5 * gross_turnover, 'linear_transaction_cost': linear_cost, 'quadratic_impact_cost': impact_cost, 'total_trading_cost': total_cost, 'concentration_hhi': concentration, 'effective_holdings': effective_holdings, 'maximum_asset_weight': float(weights.max()), 'nonzero_holdings': int(np.count_nonzero(weights > 1e-08)), 'material_holdings_count': material_holdings_count, 'active_trade_count': active_trade_count, 'top5_weight_share': top5_weight_share, 'top10_weight_share': top10_weight_share, 'hard_guardrail_status': 'PASS' if hard_breaches.empty else 'BREACH', 'hard_breach_count': int(len(hard_breaches)), 'maximum_normalized_hard_violation': float(hard_breaches['normalized_violation'].max()) if not hard_breaches.empty else 0.0, 'binding_guardrail_count': int(audit['binding_policy_guardrail'].sum()), 'mathematical_active_bound_count': int(audit['mathematical_active_bound'].sum()), 'trivial_nonnegativity_bound_count': int(audit['trivial_nonnegativity_bound'].sum()), 'guardrail_headroom_min': headroom_min, 'guardrail_headroom_10pct': headroom_10, 'guardrail_headroom_median': headroom_median, 'warning_breach_count': int(len(warning_breaches)), 'maximum_warning_excess': float(warning_breaches['warning_excess'].max()) if not warning_breaches.empty else 0.0, 'total_warning_excess': float(scenario['warning_excess'].sum()), 'minimum_warning_headroom': float(scenario['warning_headroom'].min()), 'minimum_hard_limit_headroom': float(scenario['hard_limit_headroom'].min()), 'expected_return_top5_abs_coverage': _absolute_top_coverage(asset_attr['expected_return_contribution'].to_numpy(dtype=float)), 'volatility_top5_abs_coverage': _absolute_top_coverage(asset_attr['volatility_contribution'].to_numpy(dtype=float)), 'turnover_top5_coverage': _absolute_top_coverage(asset_attr['turnover_contribution'].to_numpy(dtype=float)), 'worst_scenario_top5_abs_coverage': _absolute_top_coverage(asset_attr['worst_scenario_loss_contribution'].to_numpy(dtype=float)), **path_metrics}
    reconciliation_errors = {'expected_return': abs(float(asset_attr['expected_return_contribution'].sum()) - expected_return), 'volatility': abs(float(asset_attr['volatility_contribution'].sum()) - volatility), 'turnover': abs(float(asset_attr['turnover_contribution'].sum()) - gross_turnover), 'linear_cost': abs(float(asset_attr['linear_cost_contribution'].sum()) - linear_cost), 'impact_cost': abs(float(asset_attr['impact_cost_contribution'].sum()) - impact_cost), 'worst_scenario': abs(float(asset_attr['worst_scenario_loss_contribution'].sum()) - row['worst_scenario_loss'])}
    row['maximum_attribution_reconciliation_error'] = max(reconciliation_errors.values())
    return (row, audit, scenario, attributions)

def compare_candidates(*, context: Step6Context, candidates: Mapping[str, Mapping[str, Any]]) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, pd.DataFrame], dict[str, pd.DataFrame], dict[str, dict[str, pd.DataFrame]]]:
    context.validate()
    if not candidates:
        raise ValueError('At least one candidate is required.')
    rows: list[dict[str, Any]] = []
    weight_columns: dict[str, np.ndarray] = {}
    audits: dict[str, pd.DataFrame] = {}
    warnings: dict[str, pd.DataFrame] = {}
    attributions: dict[str, dict[str, pd.DataFrame]] = {}
    for label, candidate in candidates.items():
        row, audit, warning, attribution = _evaluate_candidate(context=context, candidate=candidate)
        rows.append(row)
        weight_columns[label] = _array(candidate['weights'])
        audits[label] = audit
        warnings[label] = warning
        attributions[label] = attribution
    comparison = pd.DataFrame(rows).set_index('candidate')
    weights = pd.DataFrame(weight_columns, index=list(context.portfolio_data.tickers))
    return (comparison, weights, audits, warnings, attributions)

def mark_duplicate_portfolios(*, comparison: pd.DataFrame, weights: pd.DataFrame, priority: Sequence[str] | None=None, tolerance: float=1e-08) -> pd.DataFrame:
    result = comparison.copy()
    labels = list(result.index)
    if priority is None:
        ordered = labels
    else:
        priority_order = {label: rank for rank, label in enumerate(priority)}
        ordered = sorted(labels, key=lambda label: (priority_order.get(label, len(priority_order)), labels.index(label)))
    canonical: list[str] = []
    duplicate_of: dict[str, str | None] = {}
    for label in ordered:
        vector = weights[label].to_numpy(dtype=float)
        match = None
        for existing in canonical:
            if np.max(np.abs(vector - weights[existing].to_numpy(dtype=float))) <= tolerance:
                match = existing
                break
        if match is None:
            canonical.append(label)
            duplicate_of[label] = None
        else:
            duplicate_of[label] = match
    result['duplicate_of'] = pd.Series(duplicate_of).reindex(result.index)
    result['is_unique_portfolio'] = result['duplicate_of'].isna()
    result['policy_compliant'] = result['hard_breach_count'].eq(0)
    result['eligible_for_selection'] = result['decision_eligible_declared'] & result['policy_compliant'] & result['is_unique_portfolio']
    return result

def _minmax_higher(series: pd.Series) -> pd.Series:
    values = series.astype(float)
    minimum = float(values.min())
    maximum = float(values.max())
    if maximum - minimum <= EPS:
        return pd.Series(0.5, index=values.index, dtype=float)
    return (values - minimum) / (maximum - minimum)

def _minmax_lower(series: pd.Series) -> pd.Series:
    return 1.0 - _minmax_higher(series)
DEFAULT_DECISION_SCENARIOS: dict[str, dict[str, float]] = {'Balanced': {'expected_return_score': 0.25, 'risk_control_score': 0.25, 'implementation_score': 0.2, 'guardrail_resilience_score': 0.2, 'explainability_score': 0.1}, 'Return First': {'expected_return_score': 0.45, 'risk_control_score': 0.2, 'implementation_score': 0.1, 'guardrail_resilience_score': 0.15, 'explainability_score': 0.1}, 'Risk First': {'expected_return_score': 0.15, 'risk_control_score': 0.45, 'implementation_score': 0.1, 'guardrail_resilience_score': 0.2, 'explainability_score': 0.1}, 'Implementation First': {'expected_return_score': 0.15, 'risk_control_score': 0.15, 'implementation_score': 0.45, 'guardrail_resilience_score': 0.15, 'explainability_score': 0.1}, 'Governance First': {'expected_return_score': 0.15, 'risk_control_score': 0.2, 'implementation_score': 0.1, 'guardrail_resilience_score': 0.45, 'explainability_score': 0.1}, 'Explainability First': {'expected_return_score': 0.15, 'risk_control_score': 0.15, 'implementation_score': 0.15, 'guardrail_resilience_score': 0.15, 'explainability_score': 0.4}}

def rank_candidates(*, comparison: pd.DataFrame, decision_scenarios: Mapping[str, Mapping[str, float]] | None=None) -> tuple[pd.DataFrame, pd.DataFrame]:
    scenarios = dict(decision_scenarios or DEFAULT_DECISION_SCENARIOS)
    ranking = comparison.copy()
    eligible = ranking.loc[ranking['eligible_for_selection']].copy()
    if eligible.empty:
        raise RuntimeError('No unique, hard-compliant decision candidate is available.')
    eligible['expected_return_score'] = _minmax_higher(eligible['expected_total_return'])
    risk_components = pd.DataFrame({'volatility': _minmax_lower(eligible['volatility']), 'scenario': _minmax_lower(eligible['worst_scenario_loss']), 'drawdown': _minmax_lower(eligible['in_sample_maximum_drawdown']), 'cvar': _minmax_lower(eligible['daily_cvar_95'])})
    eligible['risk_control_score'] = 0.35 * risk_components['volatility'] + 0.35 * risk_components['scenario'] + 0.15 * risk_components['drawdown'] + 0.15 * risk_components['cvar']
    implementation_components = pd.DataFrame({'turnover': _minmax_lower(eligible['gross_turnover']), 'cost': _minmax_lower(eligible['total_trading_cost']), 'trades': _minmax_lower(eligible['active_trade_count'])})
    eligible['implementation_score'] = 0.55 * implementation_components['turnover'] + 0.35 * implementation_components['cost'] + 0.1 * implementation_components['trades']
    guardrail_components = pd.DataFrame({'policy_headroom': _minmax_higher(eligible['guardrail_headroom_10pct'].fillna(0.0)), 'hard_scenario_headroom': _minmax_higher(eligible['minimum_hard_limit_headroom']), 'warning_excess': _minmax_lower(eligible['total_warning_excess']), 'bindings': _minmax_lower(eligible['binding_guardrail_count'])})
    eligible['guardrail_resilience_score'] = 0.35 * guardrail_components['policy_headroom'] + 0.3 * guardrail_components['hard_scenario_headroom'] + 0.25 * guardrail_components['warning_excess'] + 0.1 * guardrail_components['bindings']
    explainability_components = pd.DataFrame({'method': eligible['method_traceability'], 'holdings': _minmax_lower(eligible['material_holdings_count']), 'trades': _minmax_lower(eligible['active_trade_count']), 'coverage': eligible[['expected_return_top5_abs_coverage', 'volatility_top5_abs_coverage', 'turnover_top5_coverage', 'worst_scenario_top5_abs_coverage']].mean(axis=1)})
    eligible['explainability_score'] = 0.35 * explainability_components['method'] + 0.2 * explainability_components['holdings'] + 0.2 * explainability_components['trades'] + 0.25 * explainability_components['coverage']
    score_columns = ['expected_return_score', 'risk_control_score', 'implementation_score', 'guardrail_resilience_score', 'explainability_score']
    scenario_score_columns: dict[str, str] = {}
    scenario_rank_columns: dict[str, str] = {}
    for scenario_name, weights in scenarios.items():
        missing = set(score_columns) - set(weights)
        if missing:
            raise ValueError(f'{scenario_name}: missing decision weights for {sorted(missing)}')
        total = float(sum((float(weights[key]) for key in score_columns)))
        if abs(total - 1.0) > 1e-10:
            raise ValueError(f'{scenario_name}: weights must sum to one.')
        score_column = 'decision_score__' + scenario_name.lower().replace(' ', '_')
        rank_column = 'decision_rank__' + scenario_name.lower().replace(' ', '_')
        eligible[score_column] = sum((float(weights[key]) * eligible[key] for key in score_columns))
        eligible[rank_column] = eligible[score_column].rank(ascending=False, method='min')
        scenario_score_columns[scenario_name] = score_column
        scenario_rank_columns[scenario_name] = rank_column
    score_matrix = eligible[list(scenario_score_columns.values())]
    rank_matrix = eligible[list(scenario_rank_columns.values())]
    eligible['robust_mean_score'] = score_matrix.mean(axis=1)
    eligible['robust_mean_rank'] = rank_matrix.mean(axis=1)
    eligible['rank_best'] = rank_matrix.min(axis=1)
    eligible['rank_worst'] = rank_matrix.max(axis=1)
    eligible['top_1_frequency'] = rank_matrix.eq(1.0).mean(axis=1)
    eligible['top_3_frequency'] = rank_matrix.le(3.0).mean(axis=1)
    eligible['base_decision_score'] = eligible[scenario_score_columns['Balanced']]
    eligible['base_rank'] = eligible[scenario_rank_columns['Balanced']]
    eligible['selection_rank'] = eligible['robust_mean_score'].rank(ascending=False, method='min')
    ranking_columns = score_columns + list(scenario_score_columns.values()) + list(scenario_rank_columns.values()) + ['robust_mean_score', 'robust_mean_rank', 'rank_best', 'rank_worst', 'top_1_frequency', 'top_3_frequency', 'base_decision_score', 'base_rank', 'selection_rank']
    for column in ranking_columns:
        ranking[column] = np.nan
    ranking.loc[eligible.index, ranking_columns] = eligible[ranking_columns]
    scenario_table = pd.DataFrame({scenario_name: eligible[score_column] for scenario_name, score_column in scenario_score_columns.items()})
    return (ranking, scenario_table)

def _pareto_mask(data: pd.DataFrame, *, maximize: Sequence[str], minimize: Sequence[str]) -> pd.Series:
    labels = list(data.index)
    mask = pd.Series(True, index=labels, dtype=bool)
    for label in labels:
        row = data.loc[label]
        for other_label in labels:
            if other_label == label:
                continue
            other = data.loc[other_label]
            weakly_better = True
            strictly_better = False
            for column in maximize:
                if other[column] < row[column] - EPS:
                    weakly_better = False
                    break
                if other[column] > row[column] + EPS:
                    strictly_better = True
            if not weakly_better:
                continue
            for column in minimize:
                if other[column] > row[column] + EPS:
                    weakly_better = False
                    break
                if other[column] < row[column] - EPS:
                    strictly_better = True
            if weakly_better and strictly_better:
                mask.loc[label] = False
                break
    return mask

def add_pareto_flags(comparison: pd.DataFrame) -> pd.DataFrame:
    result = comparison.copy()
    eligible = result.loc[result['eligible_for_selection']]
    for column in ['pareto_risk_return', 'pareto_return_implementation', 'pareto_governance', 'pareto_comprehensive']:
        result[column] = False
    if eligible.empty:
        return result
    result.loc[eligible.index, 'pareto_risk_return'] = _pareto_mask(eligible, maximize=['expected_total_return'], minimize=['volatility', 'worst_scenario_loss'])
    result.loc[eligible.index, 'pareto_return_implementation'] = _pareto_mask(eligible, maximize=['expected_total_return'], minimize=['gross_turnover', 'total_trading_cost'])
    result.loc[eligible.index, 'pareto_governance'] = _pareto_mask(eligible, maximize=['minimum_hard_limit_headroom', 'guardrail_headroom_10pct'], minimize=['warning_breach_count', 'total_warning_excess'])
    result.loc[eligible.index, 'pareto_comprehensive'] = _pareto_mask(eligible, maximize=['expected_total_return'], minimize=['volatility', 'worst_scenario_loss', 'gross_turnover', 'total_warning_excess'])
    return result

def build_explainability_summary(comparison: pd.DataFrame) -> pd.DataFrame:
    columns = ['family', 'role', 'method', 'method_traceability', 'nonzero_holdings', 'material_holdings_count', 'active_trade_count', 'top5_weight_share', 'top10_weight_share', 'expected_return_top5_abs_coverage', 'volatility_top5_abs_coverage', 'turnover_top5_coverage', 'worst_scenario_top5_abs_coverage', 'binding_guardrail_count', 'maximum_attribution_reconciliation_error']
    return comparison[columns].copy()

def build_narratives(comparison: pd.DataFrame) -> pd.DataFrame:
    records: list[dict[str, str]] = []
    for label, row in comparison.iterrows():
        if row['hard_breach_count'] == 0:
            governance = 'All hard guardrails pass'
        else:
            governance = f"{int(row['hard_breach_count'])} hard guardrail breach(es)"
        if row['warning_breach_count'] == 0:
            governance += '; all scenario warnings pass.'
        else:
            governance += f"; {int(row['warning_breach_count'])} soft scenario warning breach(es)."
        implementation = f"Gross turnover {row['gross_turnover']:.2%}, estimated trading cost {row['total_trading_cost']:.4%}, {int(row['active_trade_count'])} material trade(s)."
        risk = f"Expected volatility {row['volatility']:.2%}, worst modeled scenario loss {row['worst_scenario_loss']:.2%}, in-sample maximum drawdown {row['in_sample_maximum_drawdown']:.2%}."
        explainability = f"{row['method']} Method traceability {row['method_traceability']:.0%}; {int(row['material_holdings_count'])} material holdings; top-five return attribution coverage {row['expected_return_top5_abs_coverage']:.1%}."
        records.append({'candidate': label, 'return_summary': f"Model-implied expected total return {row['expected_total_return']:.2%} ({row['expected_growth']:.2%} growth + {row['income_yield']:.2%} income).", 'risk_summary': risk, 'implementation_summary': implementation, 'governance_summary': governance, 'explainability_summary': explainability})
    return pd.DataFrame(records).set_index('candidate')

def concatenate_audits(audits: Mapping[str, pd.DataFrame]) -> pd.DataFrame:
    frames = []
    for candidate, frame in audits.items():
        copy = frame.copy()
        copy.insert(0, 'candidate', candidate)
        frames.append(copy)
    return pd.concat(frames, ignore_index=True)

def concatenate_warning_audits(warnings: Mapping[str, pd.DataFrame]) -> pd.DataFrame:
    frames = []
    for candidate, frame in warnings.items():
        copy = frame.copy()
        copy.insert(0, 'scenario', copy.index)
        copy.insert(0, 'candidate', candidate)
        frames.append(copy.reset_index(drop=True))
    return pd.concat(frames, ignore_index=True)


Writing step_06_comparison_final.py


## release.1 integrity-check correction

In [48]:
import sys
sys.modules.pop('step_06_comparison_final', None)
importlib.invalidate_caches()
import step_06_comparison_final as step6
step6 = importlib.reload(step6)
required_functions = ['Step6Context', 'register_candidate', 'compare_candidates', 'mark_duplicate_portfolios', 'rank_candidates', 'add_pareto_flags', 'build_explainability_summary', 'build_narratives']
missing_functions = [name for name in required_functions if not hasattr(step6, name)]
if missing_functions:
    raise ImportError('The Step 6 module is incomplete: ' + ', '.join(missing_functions))
module_path = Path(step6.__file__).resolve()
module_source = module_path.read_text(encoding='utf-8')
required_module_markers = ['def _evaluate_candidate', '"hard_breach_count"', '"warning_breach_count"', '"maximum_attribution_reconciliation_error"', '"trivial_nonnegativity_bound"', 'def compare_candidates', 'def rank_candidates', '"risk_control_score"', '"guardrail_resilience_score"', '"explainability_score"']
missing_markers = [marker for marker in required_module_markers if marker not in module_source]
if missing_markers:
    raise RuntimeError('The Step 6 module is missing required implementation markers: ' + ', '.join(missing_markers))
compare_source = inspect.getsource(step6.compare_candidates)
for marker in ['_evaluate_candidate', 'comparison', 'weights', 'audits', 'warnings', 'attributions']:
    if marker not in compare_source:
        raise RuntimeError(f'The Step 6 comparison wrapper is missing: {marker}')
ranking_source = inspect.getsource(step6.rank_candidates)
for marker in ['eligible_for_selection', 'expected_return_score', 'risk_control_score', 'implementation_score', 'guardrail_resilience_score', 'explainability_score']:
    if marker not in ranking_source:
        raise RuntimeError(f'The Step 6 ranking implementation is missing: {marker}')
print('PASS: Step 6 module integrity check.')
print('Module:', module_path)
print('Comparison schema fields are defined in _evaluate_candidate and exposed through compare_candidates.')


PASS: Step 6 module integrity check.
Module: /content/step_06_comparison_final.py
Comparison schema fields are defined in _evaluate_candidate and exposed through compare_candidates.


## Step 6A — Register one clean candidate universe

In [49]:
STEP6_CONTEXT = step6.Step6Context(step4=step4, portfolio_data=portfolio_data, current_weights=current_weights, scenarios=PRIMARY_SCENARIOS, constraints=constraints, trading_config=trading_config, daily_returns=STEP5_DAILY_RETURNS, hard_tolerance=5e-06, binding_tolerance=1e-05, duplicate_tolerance=1e-08, material_weight_threshold=0.005, material_trade_threshold=0.001)
STEP6_CONTEXT.validate()
STEP6_CANDIDATES = {}
step6.register_candidate(STEP6_CANDIDATES, context=STEP6_CONTEXT, label='Baseline | Current portfolio', family='Baseline', role='Incumbent benchmark', method='Current strategic allocation', weights=current_weights, source_name='current_portfolio', decision_eligible=False, independent_selection=True, method_traceability=1.0)
for stage_result in stages:
    stage_number = stage_result.stage.split('_', 1)[0]
    stage_role = 'Investable Step 4 benchmark' if stage_number in {'05', '06'} else 'Constraint-ladder diagnostic'
    step6.register_candidate(STEP6_CANDIDATES, context=STEP6_CONTEXT, label='Step 4 | ' + stage_result.stage, family='Step 4', role=stage_role, method='Deterministic classical constraint ladder', weights=stage_result.weights, source_name=stage_result.stage, decision_eligible=False, independent_selection=True, method_traceability=1.0)
STEP6_METHOD_DEFINITIONS = {'Primary unrestricted classical': {'role': 'Primary continuous benchmark', 'method': 'Full continuous classical optimization', 'eligible': True, 'independent': True, 'traceability': 1.0}, 'Independent Qiskit QAOA': {'role': 'Quantum-assisted decision candidate', 'method': 'QAOA active-set selection plus continuous refinement', 'eligible': True, 'independent': True, 'traceability': 0.65}, 'Independent exact active-set benchmark': {'role': 'Exact reduced-problem benchmark', 'method': 'Exact active-set enumeration plus continuous refinement', 'eligible': True, 'independent': True, 'traceability': 0.95}, 'Classical strict-warning reference': {'role': 'Alternative risk-governance policy', 'method': 'Continuous classical optimization with warning limits hardened', 'eligible': True, 'independent': True, 'traceability': 1.0}, 'Classical subset baseline: greedy': {'role': 'Classical active-set benchmark', 'method': 'Greedy active-set selection plus continuous refinement', 'eligible': True, 'independent': True, 'traceability': 0.9}, 'Classical subset baseline: local_search': {'role': 'Classical active-set benchmark', 'method': 'Local-search active-set selection plus continuous refinement', 'eligible': True, 'independent': True, 'traceability': 0.9}, 'Classical-target-recovery exact audit': {'role': 'Diagnostic-only target-recovery audit', 'method': 'Exact recovery of an already solved classical trade target', 'eligible': False, 'independent': False, 'traceability': 0.8}}
for profile_name, profile in STEP5_PROFILE_RESULTS.items():
    definition = STEP6_METHOD_DEFINITIONS.get(profile_name, {'role': 'Additional Step 5 profile', 'method': 'Classical Step 5 profile', 'eligible': True, 'independent': True, 'traceability': 0.9})
    metadata = {'selected_tickers': profile.get('selected_tickers', []), 'profile_name': profile.get('profile', profile_name)}
    step6.register_candidate(STEP6_CANDIDATES, context=STEP6_CONTEXT, label=profile_name, family='Step 5 / 5Q', role=definition['role'], method=definition['method'], weights=profile['result'].weights, source_name=profile_name, decision_eligible=definition['eligible'], independent_selection=definition['independent'], method_traceability=definition['traceability'], metadata=metadata)
STEP6_CANDIDATE_REGISTRY = pd.DataFrame([{'candidate': label, 'family': candidate['family'], 'role': candidate['role'], 'method': candidate['method'], 'decision_eligible_declared': candidate['decision_eligible'], 'independent_selection': candidate['independent_selection'], 'method_traceability': candidate['method_traceability']} for label, candidate in STEP6_CANDIDATES.items()]).set_index('candidate')
print('Registered Step 6 candidates:', len(STEP6_CANDIDATES))
display(STEP6_CANDIDATE_REGISTRY)


Registered Step 6 candidates: 14


,family,role,method,decision_eligible_declared,independent_selection,method_traceability
candidate,,,,,,
Baseline | Current portfolio,Baseline,Incumbent benchmark,Current strategic allocation,False,True,1.00
Step 4 | 01_minimum_variance,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,False,True,1.00
Step 4 | 02_mean_variance,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,False,True,1.00
Step 4 | 03_asset_caps,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,False,True,1.00
Step 4 | 04_guardrails,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,False,True,1.00
Step 4 | 05_trading_costs,Step 4,Investable Step 4 benchmark,Deterministic classical constraint ladder,False,True,1.00
Step 4 | 06_scenario_aware,Step 4,Investable Step 4 benchmark,Deterministic classical constraint ladder,False,True,1.00
Primary unrestricted classical,Step 5 / 5Q,Primary continuous benchmark,Full continuous classical optimization,True,True,1.00
Independent Qiskit QAOA,Step 5 / 5Q,Quantum-assisted decision candidate,QAOA active-set selection plus continuous refi...,True,True,0.65


## Step 6B — Re-evaluate every portfolio with one measuring stick

In [50]:
STEP6_COMPARISON, STEP6_WEIGHT_MATRIX, STEP6_AUDITS, STEP6_WARNING_AUDITS, STEP6_ATTRIBUTIONS = step6.compare_candidates(context=STEP6_CONTEXT, candidates=STEP6_CANDIDATES)
STEP6_FORWARD_NAME_MAP = {'Primary unrestricted classical': 'Primary classical', 'Independent Qiskit QAOA': 'Independent QAOA', 'Independent exact active-set benchmark': 'Independent exact active set', 'Classical strict-warning reference': 'Strict-warning classical'}
forward_columns = {'paths': 'forward_paths', 'median_return': 'forward_median_return', 'return_05': 'forward_return_05', 'median_volatility': 'forward_median_volatility', 'median_maximum_drawdown': 'forward_median_maximum_drawdown', 'drawdown_95': 'forward_drawdown_95', 'loss_path_frequency': 'forward_loss_path_frequency'}
for output_column in forward_columns.values():
    STEP6_COMPARISON[output_column] = np.nan
if 'FORWARD_SIMULATION_SUMMARY' in globals() and isinstance(FORWARD_SIMULATION_SUMMARY, pd.DataFrame) and (not FORWARD_SIMULATION_SUMMARY.empty):
    for candidate_name, forward_name in STEP6_FORWARD_NAME_MAP.items():
        if candidate_name in STEP6_COMPARISON.index and forward_name in FORWARD_SIMULATION_SUMMARY.index:
            for source_column, output_column in forward_columns.items():
                STEP6_COMPARISON.loc[candidate_name, output_column] = FORWARD_SIMULATION_SUMMARY.loc[forward_name, source_column]
STEP6_COMMON_METRICS = STEP6_COMPARISON[['family', 'role', 'expected_growth', 'income_yield', 'expected_total_return', 'volatility', 'return_to_volatility', 'worst_scenario_loss', 'in_sample_maximum_drawdown', 'daily_var_95', 'daily_cvar_95', 'gross_turnover', 'total_trading_cost', 'effective_holdings', 'maximum_asset_weight', 'active_trade_count', 'hard_guardrail_status', 'hard_breach_count', 'warning_breach_count', 'minimum_hard_limit_headroom']].sort_values(['hard_breach_count', 'expected_total_return'], ascending=[True, False])
display(STEP6_COMMON_METRICS.style.format({'expected_growth': '{:.2%}', 'income_yield': '{:.2%}', 'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'return_to_volatility': '{:.3f}', 'worst_scenario_loss': '{:.2%}', 'in_sample_maximum_drawdown': '{:.2%}', 'daily_var_95': '{:.2%}', 'daily_cvar_95': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'effective_holdings': '{:.2f}', 'maximum_asset_weight': '{:.2%}', 'active_trade_count': '{:.0f}', 'hard_breach_count': '{:.0f}', 'warning_breach_count': '{:.0f}', 'minimum_hard_limit_headroom': '{:.2%}'}))
print('Path interpretation:', 'synthetic in-sample diagnostics' if DATA_SOURCE == 'synthetic' else 'historical in-sample diagnostics')


,family,role,expected_growth,income_yield,expected_total_return,volatility,return_to_volatility,worst_scenario_loss,in_sample_maximum_drawdown,daily_var_95,daily_cvar_95,gross_turnover,total_trading_cost,effective_holdings,maximum_asset_weight,active_trade_count,hard_guardrail_status,hard_breach_count,warning_breach_count,minimum_hard_limit_headroom
candidate,,,,,,,,,,,,,,,,,,,,
Baseline | Current portfolio,Baseline,Incumbent benchmark,3.14%,2.40%,5.55%,8.82%,0.628,13.83%,8.06%,0.84%,1.12%,0.00%,0.0000%,40.14,5.00%,0,PASS,0,5,2.50%
Primary unrestricted classical,Step 5 / 5Q,Primary continuous benchmark,2.90%,2.59%,5.49%,7.86%,0.698,12.71%,7.11%,0.74%,0.99%,13.09%,0.0020%,33.73,7.97%,6,PASS,0,5,2.96%
Classical-target-recovery exact audit,Step 5 / 5Q,Diagnostic-only target-recovery audit,2.90%,2.59%,5.49%,7.86%,0.698,12.71%,7.11%,0.74%,0.99%,13.09%,0.0020%,33.73,7.97%,6,PASS,0,5,2.96%
Classical strict-warning reference,Step 5 / 5Q,Alternative risk-governance policy,2.82%,2.67%,5.48%,7.51%,0.731,12.17%,6.53%,0.72%,0.95%,23.88%,0.0056%,29.35,10.19%,10,PASS,0,0,4.00%
Independent Qiskit QAOA,Step 5 / 5Q,Quantum-assisted decision candidate,2.90%,2.56%,5.45%,7.87%,0.693,12.72%,7.13%,0.74%,1.00%,11.06%,0.0015%,35.48,6.96%,4,PASS,0,5,2.95%
Independent exact active-set benchmark,Step 5 / 5Q,Exact reduced-problem benchmark,2.90%,2.56%,5.45%,7.87%,0.693,12.72%,7.13%,0.74%,1.00%,11.06%,0.0015%,35.48,6.96%,4,PASS,0,5,2.95%
Classical subset baseline: greedy,Step 5 / 5Q,Classical active-set benchmark,2.90%,2.56%,5.45%,7.87%,0.693,12.72%,7.13%,0.74%,1.00%,11.06%,0.0015%,35.48,6.96%,4,PASS,0,5,2.95%
Classical subset baseline: local_search,Step 5 / 5Q,Classical active-set benchmark,2.90%,2.56%,5.45%,7.87%,0.693,12.72%,7.13%,0.74%,1.00%,11.06%,0.0015%,35.48,6.96%,4,PASS,0,5,2.95%
Step 4 | 05_trading_costs,Step 4,Investable Step 4 benchmark,2.64%,2.79%,5.44%,6.00%,0.906,10.48%,5.56%,0.57%,0.75%,50.00%,0.0198%,20.19,13.43%,26,PASS,0,2,2.79%


Path interpretation: synthetic in-sample diagnostics


## Step 6C — Guardrails, hard breaches, soft warnings, and headroom

In [51]:
STEP6_ALL_AUDITS = step6.concatenate_audits(STEP6_AUDITS)
STEP6_ALL_WARNING_AUDITS = step6.concatenate_warning_audits(STEP6_WARNING_AUDITS)
STEP6_BREACH_DETAIL = STEP6_ALL_AUDITS.loc[~STEP6_ALL_AUDITS['satisfied']].sort_values('normalized_violation', ascending=False).reset_index(drop=True)
STEP6_BINDING_GUARDRAILS = STEP6_ALL_AUDITS.loc[STEP6_ALL_AUDITS['binding_policy_guardrail']].sort_values(['candidate', 'category', 'constraint']).reset_index(drop=True)
STEP6_WARNING_BREACH_DETAIL = STEP6_ALL_WARNING_AUDITS.loc[~STEP6_ALL_WARNING_AUDITS['warning_satisfied']].sort_values(['candidate', 'warning_excess'], ascending=[True, False]).reset_index(drop=True)
STEP6_GUARDRAIL_SUMMARY = STEP6_COMPARISON[['family', 'role', 'hard_guardrail_status', 'hard_breach_count', 'maximum_normalized_hard_violation', 'binding_guardrail_count', 'mathematical_active_bound_count', 'trivial_nonnegativity_bound_count', 'guardrail_headroom_min', 'guardrail_headroom_10pct', 'guardrail_headroom_median', 'warning_breach_count', 'maximum_warning_excess', 'total_warning_excess', 'minimum_warning_headroom', 'minimum_hard_limit_headroom']].sort_values(['hard_breach_count', 'warning_breach_count', 'guardrail_headroom_10pct'], ascending=[True, True, False])
display(STEP6_GUARDRAIL_SUMMARY.style.format({'hard_breach_count': '{:.0f}', 'maximum_normalized_hard_violation': '{:.2%}', 'binding_guardrail_count': '{:.0f}', 'mathematical_active_bound_count': '{:.0f}', 'trivial_nonnegativity_bound_count': '{:.0f}', 'guardrail_headroom_min': '{:.2%}', 'guardrail_headroom_10pct': '{:.2%}', 'guardrail_headroom_median': '{:.2%}', 'warning_breach_count': '{:.0f}', 'maximum_warning_excess': '{:.4%}', 'total_warning_excess': '{:.4%}', 'minimum_warning_headroom': '{:.4%}', 'minimum_hard_limit_headroom': '{:.2%}'}))
print('Detailed hard-policy breaches:', len(STEP6_BREACH_DETAIL))
if not STEP6_BREACH_DETAIL.empty:
    display(STEP6_BREACH_DETAIL[['candidate', 'category', 'constraint', 'value', 'lower', 'upper', 'absolute_violation', 'normalized_violation']].style.format({'value': '{:.6f}', 'lower': '{:.6f}', 'upper': '{:.6f}', 'absolute_violation': '{:.4%}', 'normalized_violation': '{:.2%}'}))
print('Economically meaningful binding guardrails:', len(STEP6_BINDING_GUARDRAILS))
if not STEP6_BINDING_GUARDRAILS.empty:
    display(STEP6_BINDING_GUARDRAILS[['candidate', 'category', 'constraint', 'value', 'lower', 'upper', 'policy_normalized_margin']].style.format({'value': '{:.6f}', 'lower': '{:.6f}', 'upper': '{:.6f}', 'policy_normalized_margin': '{:.2%}'}))
_invalid_zero_class_bindings = STEP6_BINDING_GUARDRAILS.loc[STEP6_BINDING_GUARDRAILS['category'].eq('asset_class') & np.isclose(STEP6_BINDING_GUARDRAILS['lower'], 0.0) & (STEP6_BINDING_GUARDRAILS['value'].abs() <= STEP6_CONTEXT.binding_tolerance)]
if not _invalid_zero_class_bindings.empty:
    raise AssertionError('Zero-weight asset classes with zero minimums were incorrectly classified as binding policy guardrails.')
print('PASS: zero-weight asset and asset-class nonnegativity bounds are excluded from economic binding counts.')
print('Soft scenario-warning breaches:', len(STEP6_WARNING_BREACH_DETAIL))
if not STEP6_WARNING_BREACH_DETAIL.empty:
    display(STEP6_WARNING_BREACH_DETAIL[['candidate', 'scenario', 'scenario_loss', 'warning_threshold', 'warning_excess', 'hard_limit', 'hard_limit_headroom']].style.format({'scenario_loss': '{:.4%}', 'warning_threshold': '{:.4%}', 'warning_excess': '{:.4%}', 'hard_limit': '{:.4%}', 'hard_limit_headroom': '{:.4%}'}))


,family,role,hard_guardrail_status,hard_breach_count,maximum_normalized_hard_violation,binding_guardrail_count,mathematical_active_bound_count,trivial_nonnegativity_bound_count,guardrail_headroom_min,guardrail_headroom_10pct,guardrail_headroom_median,warning_breach_count,maximum_warning_excess,total_warning_excess,minimum_warning_headroom,minimum_hard_limit_headroom
candidate,,,,,,,,,,,,,,,,
Classical strict-warning reference,Step 5 / 5Q,Alternative risk-governance policy,PASS,0,0.00%,0,10,7,2.06%,10.74%,40.83%,0,0.0000%,0.0000%,-0.0000%,4.00%
Step 4 | 06_scenario_aware,Step 4,Investable Step 4 benchmark,PASS,0,0.00%,4,19,12,-0.00%,9.60%,42.70%,1,0.1154%,0.1154%,-0.1154%,3.88%
Step 4 | 05_trading_costs,Step 4,Investable Step 4 benchmark,PASS,0,0.00%,3,20,14,-0.00%,9.87%,43.50%,2,1.2124%,1.5153%,-1.2124%,2.79%
Primary unrestricted classical,Step 5 / 5Q,Primary continuous benchmark,PASS,0,0.00%,0,8,5,9.75%,11.24%,40.00%,5,1.0433%,2.6529%,-1.0433%,2.96%
Classical-target-recovery exact audit,Step 5 / 5Q,Diagnostic-only target-recovery audit,PASS,0,0.00%,0,8,5,9.75%,11.24%,40.00%,5,1.0433%,2.6529%,-1.0433%,2.96%
Independent Qiskit QAOA,Step 5 / 5Q,Quantum-assisted decision candidate,PASS,0,0.00%,0,5,2,4.90%,10.44%,40.00%,5,1.0467%,2.6434%,-1.0467%,2.95%
Independent exact active-set benchmark,Step 5 / 5Q,Exact reduced-problem benchmark,PASS,0,0.00%,0,5,2,4.90%,10.44%,40.00%,5,1.0467%,2.6434%,-1.0467%,2.95%
Classical subset baseline: greedy,Step 5 / 5Q,Classical active-set benchmark,PASS,0,0.00%,0,5,2,4.90%,10.44%,40.00%,5,1.0467%,2.6434%,-1.0467%,2.95%
Classical subset baseline: local_search,Step 5 / 5Q,Classical active-set benchmark,PASS,0,0.00%,0,5,2,4.90%,10.44%,40.00%,5,1.0467%,2.6434%,-1.0467%,2.95%


Detailed hard-policy breaches: 26


,candidate,category,constraint,value,lower,upper,absolute_violation,normalized_violation
0,Step 4 | 01_minimum_variance,asset_class,class_Cash,0.976755,0.010000,0.150000,82.6755%,84.64%
1,Step 4 | 02_mean_variance,asset_class,class_Cash,0.649505,0.010000,0.150000,49.9505%,76.91%
2,Step 4 | 02_mean_variance,asset,weight_SGOV,0.649505,0.000000,0.150000,49.9505%,76.91%
3,Step 4 | 01_minimum_variance,trading,gross_turnover,1.905944,-inf,0.500000,140.5944%,73.77%
4,Step 4 | 02_mean_variance,trading,gross_turnover,1.780234,-inf,0.500000,128.0234%,71.91%
5,Step 4 | 01_minimum_variance,asset,weight_SGOV,0.508836,0.000000,0.150000,35.8836%,70.52%
6,Step 4 | 01_minimum_variance,asset,weight_BIL,0.467920,0.000000,0.150000,31.7920%,67.94%
7,Step 4 | 03_asset_caps,trading,gross_turnover,1.334085,-inf,0.500000,83.4085%,62.52%
8,Step 4 | 04_guardrails,trading,gross_turnover,1.105986,-inf,0.500000,60.5986%,54.79%
9,Step 4 | 02_mean_variance,asset,weight_MUB,0.179158,0.000000,0.100000,7.9158%,44.18%


Economically meaningful binding guardrails: 20


,candidate,category,constraint,value,lower,upper,policy_normalized_margin
0,Step 4 | 03_asset_caps,asset,weight_AGG,0.100000,0.000000,0.100000,0.00%
1,Step 4 | 03_asset_caps,asset,weight_BWX,0.100000,0.000000,0.100000,0.00%
2,Step 4 | 03_asset_caps,asset,weight_DBMF,0.100000,0.000000,0.100000,0.00%
3,Step 4 | 03_asset_caps,asset,weight_MUB,0.100000,0.000000,0.100000,0.00%
4,Step 4 | 03_asset_caps,asset,weight_SGOV,0.150000,0.000000,0.150000,0.00%
5,Step 4 | 03_asset_caps,asset_class,class_Alternatives,0.100000,0.000000,0.100000,-0.00%
6,Step 4 | 04_guardrails,asset,weight_BWX,0.100000,0.000000,0.100000,0.00%
7,Step 4 | 04_guardrails,asset,weight_MUB,0.100000,0.000000,0.100000,0.00%
8,Step 4 | 04_guardrails,asset,weight_SGOV,0.150000,0.000000,0.150000,0.00%
9,Step 4 | 04_guardrails,asset_class,class_Cash,0.150000,0.010000,0.150000,-0.00%


PASS: zero-weight asset and asset-class nonnegativity bounds are excluded from economic binding counts.
Soft scenario-warning breaches: 43


,candidate,scenario,scenario_loss,warning_threshold,warning_excess,hard_limit,hard_limit_headroom
0,Baseline | Current portfolio,Global equity selloff,12.9101%,11.4101%,1.5000%,15.4101%,2.5000%
1,Baseline | Current portfolio,Inflation and rate shock,8.4719%,6.9719%,1.5000%,10.9719%,2.5000%
2,Baseline | Current portfolio,Credit and liquidity crisis,11.4090%,9.9090%,1.5000%,13.9090%,2.5000%
3,Baseline | Current portfolio,Commodity supply shock,3.5282%,2.0282%,1.5000%,6.0282%,2.5000%
4,Baseline | Current portfolio,Broad deleveraging shock,13.8270%,12.3270%,1.5000%,16.3270%,2.5000%
5,Classical subset baseline: greedy,Commodity supply shock,3.0750%,2.0282%,1.0467%,6.0282%,2.9533%
6,Classical subset baseline: greedy,Inflation and rate shock,7.6814%,6.9719%,0.7096%,10.9719%,3.2904%
7,Classical subset baseline: greedy,Credit and liquidity crisis,10.4030%,9.9090%,0.4940%,13.9090%,3.5060%
8,Classical subset baseline: greedy,Broad deleveraging shock,12.7160%,12.3270%,0.3890%,16.3270%,3.6110%
9,Classical subset baseline: greedy,Global equity selloff,11.4143%,11.4101%,0.0042%,15.4101%,3.9958%


## Step 6D — Exact explainability and attribution

In [52]:
STEP6_EXPLAINABILITY_SUMMARY = step6.build_explainability_summary(STEP6_COMPARISON)
STEP6_NARRATIVES = step6.build_narratives(STEP6_COMPARISON)
STEP6_ATTRIBUTION_RECONCILIATION = STEP6_COMPARISON[['maximum_attribution_reconciliation_error']].copy()
if STEP6_ATTRIBUTION_RECONCILIATION['maximum_attribution_reconciliation_error'].max() > 1e-09:
    raise AssertionError('An attribution identity failed reconciliation.')
STEP6_KEY_PROFILES = [name for name in ['Primary unrestricted classical', 'Independent Qiskit QAOA', 'Independent exact active-set benchmark', 'Classical strict-warning reference', 'Classical subset baseline: greedy', 'Classical subset baseline: local_search'] if name in STEP6_ATTRIBUTIONS]
top_attribution_records = []
top_trade_records = []
class_exposure_columns = {}
for candidate_name in STEP6_KEY_PROFILES:
    asset_table = STEP6_ATTRIBUTIONS[candidate_name]['asset']
    class_exposure_columns[candidate_name] = STEP6_ATTRIBUTIONS[candidate_name]['asset_class']['weight']
    top_trades = asset_table.loc[asset_table['absolute_trade'] >= STEP6_CONTEXT.material_trade_threshold].sort_values('absolute_trade', ascending=False).head(10)
    for ticker, row in top_trades.iterrows():
        top_trade_records.append({'candidate': candidate_name, 'ticker': ticker, 'asset_class': row['asset_class'], 'current_weight': row['current_weight'], 'weight': row['weight'], 'trade': row['trade'], 'absolute_trade': row['absolute_trade']})
    attribution_columns = {'expected_return': 'expected_return_contribution', 'volatility': 'volatility_contribution', 'turnover': 'turnover_contribution', 'trading_cost': 'total_cost_contribution', 'worst_scenario': 'worst_scenario_loss_contribution'}
    for dimension, column in attribution_columns.items():
        top = asset_table.assign(absolute_contribution=asset_table[column].abs()).sort_values('absolute_contribution', ascending=False).head(10)
        for ticker, row in top.iterrows():
            top_attribution_records.append({'candidate': candidate_name, 'dimension': dimension, 'ticker': ticker, 'asset_class': row['asset_class'], 'contribution': row[column], 'absolute_contribution': row['absolute_contribution']})
STEP6_TOP_TRADES = pd.DataFrame(top_trade_records)
STEP6_TOP_ATTRIBUTIONS = pd.DataFrame(top_attribution_records)
STEP6_CLASS_EXPOSURES = pd.DataFrame(class_exposure_columns).fillna(0.0)
display(STEP6_EXPLAINABILITY_SUMMARY.style.format({'method_traceability': '{:.0%}', 'nonzero_holdings': '{:.0f}', 'material_holdings_count': '{:.0f}', 'active_trade_count': '{:.0f}', 'top5_weight_share': '{:.1%}', 'top10_weight_share': '{:.1%}', 'expected_return_top5_abs_coverage': '{:.1%}', 'volatility_top5_abs_coverage': '{:.1%}', 'turnover_top5_coverage': '{:.1%}', 'worst_scenario_top5_abs_coverage': '{:.1%}', 'binding_guardrail_count': '{:.0f}', 'maximum_attribution_reconciliation_error': '{:.2e}'}))
print('Top material trades:')
display(STEP6_TOP_TRADES.style.format({'current_weight': '{:.2%}', 'weight': '{:.2%}', 'trade': '{:+.2%}', 'absolute_trade': '{:.2%}'}))
print('PASS: all return, risk, cost, turnover, and scenario attributions reconcile.')


,family,role,method,method_traceability,nonzero_holdings,material_holdings_count,active_trade_count,top5_weight_share,top10_weight_share,expected_return_top5_abs_coverage,volatility_top5_abs_coverage,turnover_top5_coverage,worst_scenario_top5_abs_coverage,binding_guardrail_count,maximum_attribution_reconciliation_error
candidate,,,,,,,,,,,,,,,
Baseline | Current portfolio,Baseline,Incumbent benchmark,Current strategic allocation,100%,50,49,0,20.0%,36.2%,18.5%,21.5%,100.0%,20.2%,0,2.78e-17
Step 4 | 01_minimum_variance,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,100%,11,4,49,99.5%,100.0%,99.2%,99.5%,56.3%,79.5%,0,8.67e-19
Step 4 | 02_mean_variance,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,100%,7,6,50,97.5%,100.0%,96.4%,95.8%,52.8%,96.9%,0,6.94e-18
Step 4 | 03_asset_caps,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,100%,15,15,50,55.0%,90.4%,59.1%,79.2%,32.0%,73.5%,6,1.39e-17
Step 4 | 04_guardrails,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,100%,20,19,50,51.0%,76.9%,50.4%,58.5%,35.7%,54.1%,7,2.78e-17
Step 4 | 05_trading_costs,Step 4,Investable Step 4 benchmark,Deterministic classical constraint ladder,100%,37,34,26,38.3%,56.2%,36.7%,33.2%,53.5%,32.8%,3,1.39e-17
Step 4 | 06_scenario_aware,Step 4,Investable Step 4 benchmark,Deterministic classical constraint ladder,100%,38,35,25,37.7%,55.6%,36.4%,32.6%,52.5%,32.0%,4,1.39e-17
Primary unrestricted classical,Step 5 / 5Q,Primary continuous benchmark,Full continuous classical optimization,100%,46,46,6,24.3%,41.4%,22.4%,23.8%,96.3%,21.9%,0,2.78e-17
Independent Qiskit QAOA,Step 5 / 5Q,Quantum-assisted decision candidate,QAOA active-set selection plus continuous refinement,65%,48,47,4,23.3%,40.4%,21.6%,23.7%,100.0%,21.9%,0,2.78e-17


Top material trades:


,candidate,ticker,asset_class,current_weight,weight,trade,absolute_trade
0,Primary unrestricted classical,SGOV,Cash,1.43%,7.97%,+6.55%,6.55%
1,Primary unrestricted classical,VUG,US Equity,2.37%,0.00%,-2.37%,2.37%
2,Primary unrestricted classical,QQQ,US Equity,2.06%,0.00%,-2.06%,2.06%
3,Primary unrestricted classical,MTUM,US Equity,2.26%,1.15%,-1.12%,1.12%
4,Primary unrestricted classical,FXE,FX,0.51%,0.00%,-0.51%,0.51%
5,Primary unrestricted classical,UUP,FX,0.49%,0.00%,-0.49%,0.49%
6,Independent Qiskit QAOA,SGOV,Cash,1.43%,6.96%,+5.53%,5.53%
7,Independent Qiskit QAOA,VUG,US Equity,2.37%,0.00%,-2.37%,2.37%
8,Independent Qiskit QAOA,QQQ,US Equity,2.06%,0.00%,-2.06%,2.06%
9,Independent Qiskit QAOA,MTUM,US Equity,2.26%,1.16%,-1.10%,1.10%


PASS: all return, risk, cost, turnover, and scenario attributions reconcile.


## Step 6E — Duplicate removal and robust decision ranking

In [53]:
STEP6_DUPLICATE_PRIORITY = ['Primary unrestricted classical', 'Independent Qiskit QAOA', 'Independent exact active-set benchmark', 'Classical strict-warning reference', 'Classical subset baseline: greedy', 'Classical subset baseline: local_search', 'Classical-target-recovery exact audit', 'Baseline | Current portfolio']
STEP6_COMPARISON = step6.mark_duplicate_portfolios(comparison=STEP6_COMPARISON, weights=STEP6_WEIGHT_MATRIX, priority=STEP6_DUPLICATE_PRIORITY, tolerance=STEP6_CONTEXT.duplicate_tolerance)
STEP6_RANKING, STEP6_DECISION_SCENARIO_SCORES = step6.rank_candidates(comparison=STEP6_COMPARISON)
STEP6_RANKING = step6.add_pareto_flags(STEP6_RANKING)
STEP6_SCORE_SCOPE = 'relative_to_current_unique_hard_compliant_declared_shortlist'
STEP6_RELATIVE_SCORE_ALIASES = {'expected_return_score': 'relative_expected_return_score', 'risk_control_score': 'relative_risk_control_score', 'implementation_score': 'relative_implementation_score', 'guardrail_resilience_score': 'relative_guardrail_resilience_score', 'explainability_score': 'relative_explainability_score', 'base_decision_score': 'relative_balanced_decision_score', 'robust_mean_score': 'relative_robust_mean_score'}
for source_column, alias_column in STEP6_RELATIVE_SCORE_ALIASES.items():
    STEP6_RANKING[alias_column] = STEP6_RANKING[source_column]
STEP6_DECISION_WEIGHTS = pd.DataFrame(step6.DEFAULT_DECISION_SCENARIOS).T
STEP6_SCORE_DEFINITIONS = pd.DataFrame([{'display_score': alias_column, 'source_score': source_column, 'scope': STEP6_SCORE_SCOPE, 'interpretation': 'Relative min–max score within the current eligible shortlist; not an absolute financial rating.'} for source_column, alias_column in STEP6_RELATIVE_SCORE_ALIASES.items()]).set_index('display_score')
STEP6_RANKING_VIEW = STEP6_RANKING[['family', 'role', 'method', 'hard_guardrail_status', 'warning_breach_count', 'duplicate_of', 'policy_compliant', 'eligible_for_selection', 'relative_expected_return_score', 'relative_risk_control_score', 'relative_implementation_score', 'relative_guardrail_resilience_score', 'relative_explainability_score', 'relative_balanced_decision_score', 'relative_robust_mean_score', 'robust_mean_rank', 'rank_best', 'rank_worst', 'top_1_frequency', 'top_3_frequency', 'selection_rank', 'pareto_risk_return', 'pareto_return_implementation', 'pareto_governance', 'pareto_comprehensive']].sort_values(['eligible_for_selection', 'selection_rank', 'relative_robust_mean_score'], ascending=[False, True, False])
print('Predeclared Step 6 decision weights:')
display(STEP6_DECISION_WEIGHTS.style.format('{:.0%}'))
print('Score interpretation: component and composite scores shown below are relative min–max scores within the current unique eligible shortlist.')
print('They are not absolute expected-return, risk, implementation, governance, or explainability ratings.')
display(STEP6_SCORE_DEFINITIONS)
print('Unique hard-compliant candidates included in ranking:', int(STEP6_RANKING['eligible_for_selection'].sum()))
display(STEP6_RANKING_VIEW.style.format({'warning_breach_count': '{:.0f}', 'relative_expected_return_score': '{:.1%}', 'relative_risk_control_score': '{:.1%}', 'relative_implementation_score': '{:.1%}', 'relative_guardrail_resilience_score': '{:.1%}', 'relative_explainability_score': '{:.1%}', 'relative_balanced_decision_score': '{:.3f}', 'relative_robust_mean_score': '{:.3f}', 'robust_mean_rank': '{:.2f}', 'rank_best': '{:.0f}', 'rank_worst': '{:.0f}', 'top_1_frequency': '{:.0%}', 'top_3_frequency': '{:.0%}', 'selection_rank': '{:.0f}'}))
print('Relative decision-scenario scores:')
display(STEP6_DECISION_SCENARIO_SCORES.style.format('{:.3f}'))
STEP6_SHORTLIST = STEP6_RANKING.loc[STEP6_RANKING['eligible_for_selection']].sort_values(['selection_rank', 'robust_mean_score'])
STEP6_PRIMARY_RECOMMENDATION = STEP6_SHORTLIST.index[0]
print('Top robust candidate under the declared Step 6 scenarios:', STEP6_PRIMARY_RECOMMENDATION)
print('This is a preference-dependent governance result, not a universal mathematical winner.')


Predeclared Step 6 decision weights:


,expected_return_score,risk_control_score,implementation_score,guardrail_resilience_score,explainability_score
Balanced,25%,25%,20%,20%,10%
Return First,45%,20%,10%,15%,10%
Risk First,15%,45%,10%,20%,10%
Implementation First,15%,15%,45%,15%,10%
Governance First,15%,20%,10%,45%,10%
Explainability First,15%,15%,15%,15%,40%


Score interpretation: component and composite scores shown below are relative min–max scores within the current unique eligible shortlist.
They are not absolute expected-return, risk, implementation, governance, or explainability ratings.


,source_score,scope,interpretation
display_score,,,
relative_expected_return_score,expected_return_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...
relative_risk_control_score,risk_control_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...
relative_implementation_score,implementation_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...
relative_guardrail_resilience_score,guardrail_resilience_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...
relative_explainability_score,explainability_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...
relative_balanced_decision_score,base_decision_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...
relative_robust_mean_score,robust_mean_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...


Unique hard-compliant candidates included in ranking: 3


,family,role,method,hard_guardrail_status,warning_breach_count,duplicate_of,policy_compliant,eligible_for_selection,relative_expected_return_score,relative_risk_control_score,relative_implementation_score,relative_guardrail_resilience_score,relative_explainability_score,relative_balanced_decision_score,relative_robust_mean_score,robust_mean_rank,rank_best,rank_worst,top_1_frequency,top_3_frequency,selection_rank,pareto_risk_return,pareto_return_implementation,pareto_governance,pareto_comprehensive
candidate,,,,,,,,,,,,,,,,,,,,,,,,,
Classical strict-warning reference,Step 5 / 5Q,Alternative risk-governance policy,Continuous classical optimization with warning limits hardened,PASS,0,None,True,True,85.7%,100.0%,0.0%,72.9%,65.0%,0.675,0.675,1.33,1,3,83%,100%,1,True,False,True,True
Primary unrestricted classical,Step 5 / 5Q,Primary continuous benchmark,Full continuous classical optimization,PASS,5,None,True,True,100.0%,1.7%,83.8%,40.1%,65.3%,0.567,0.559,1.83,1,2,17%,100%,2,True,True,True,True
Independent Qiskit QAOA,Step 5 / 5Q,Quantum-assisted decision candidate,QAOA active-set selection plus continuous refinement,PASS,5,None,True,True,0.0%,0.0%,100.0%,5.1%,53.2%,0.263,0.274,2.83,2,3,0%,100%,3,False,True,False,True
Baseline | Current portfolio,Baseline,Incumbent benchmark,Current strategic allocation,PASS,5,None,True,False,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan,nan,nan%,nan%,nan,False,False,False,False
Step 4 | 01_minimum_variance,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,BREACH,0,None,False,False,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan,nan,nan%,nan%,nan,False,False,False,False
Step 4 | 02_mean_variance,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,BREACH,1,None,False,False,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan,nan,nan%,nan%,nan,False,False,False,False
Step 4 | 03_asset_caps,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,BREACH,2,None,False,False,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan,nan,nan%,nan%,nan,False,False,False,False
Step 4 | 04_guardrails,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,BREACH,2,None,False,False,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan,nan,nan%,nan%,nan,False,False,False,False
Step 4 | 05_trading_costs,Step 4,Investable Step 4 benchmark,Deterministic classical constraint ladder,PASS,2,None,True,False,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan,nan,nan%,nan%,nan,False,False,False,False


Relative decision-scenario scores:


,Balanced,Return First,Risk First,Implementation First,Governance First,Explainability First
candidate,,,,,,
Primary unrestricted classical,0.567,0.663,0.387,0.655,0.483,0.599
Independent Qiskit QAOA,0.263,0.161,0.163,0.511,0.176,0.370
Classical strict-warning reference,0.675,0.760,0.789,0.453,0.722,0.648


Top robust candidate under the declared Step 6 scenarios: Classical strict-warning reference
This is a preference-dependent governance result, not a universal mathematical winner.


## Step 6E2 — Forward-model evidence sufficiency

In [54]:
STEP6_FORWARD_VALIDATION = pd.DataFrame(index=STEP6_SHORTLIST.index)
STEP6_FORWARD_VALIDATION['forward_paths'] = STEP6_SHORTLIST['forward_paths']
STEP6_FORWARD_VALIDATION['forward_median_return'] = STEP6_SHORTLIST['forward_median_return']
STEP6_FORWARD_VALIDATION['forward_return_05'] = STEP6_SHORTLIST['forward_return_05']
STEP6_FORWARD_VALIDATION['forward_median_volatility'] = STEP6_SHORTLIST['forward_median_volatility']
STEP6_FORWARD_VALIDATION['forward_median_maximum_drawdown'] = STEP6_SHORTLIST['forward_median_maximum_drawdown']
STEP6_FORWARD_VALIDATION['forward_drawdown_95'] = STEP6_SHORTLIST['forward_drawdown_95']
STEP6_FORWARD_VALIDATION['forward_loss_path_frequency'] = STEP6_SHORTLIST['forward_loss_path_frequency']

def _forward_evidence_tier(paths: float) -> str:
    if not np.isfinite(paths):
        return 'NOT_AVAILABLE'
    if paths >= 500:
        return 'FINAL_EVIDENCE'
    return 'PRELIMINARY_FAST_MODE'
STEP6_FORWARD_VALIDATION['evidence_tier'] = STEP6_FORWARD_VALIDATION['forward_paths'].apply(_forward_evidence_tier)
STEP6_FORWARD_VALIDATION['ranking_treatment'] = 'separate_validation_not_scored'
STEP6_FINAL_FORWARD_EVIDENCE_READY = bool(STEP6_FORWARD_VALIDATION['evidence_tier'].eq('FINAL_EVIDENCE').all())
STEP6_RANKING['forward_evidence_tier'] = 'NOT_APPLICABLE'
STEP6_RANKING.loc[STEP6_FORWARD_VALIDATION.index, 'forward_evidence_tier'] = STEP6_FORWARD_VALIDATION['evidence_tier']
display(STEP6_FORWARD_VALIDATION.style.format({'forward_paths': '{:.0f}', 'forward_median_return': '{:.2%}', 'forward_return_05': '{:.2%}', 'forward_median_volatility': '{:.2%}', 'forward_median_maximum_drawdown': '{:.2%}', 'forward_drawdown_95': '{:.2%}', 'forward_loss_path_frequency': '{:.1%}'}))
print('Final forward-evidence readiness:', STEP6_FINAL_FORWARD_EVIDENCE_READY)
if not STEP6_FINAL_FORWARD_EVIDENCE_READY:
    print('Development interpretation only: rerun the underlying notebook with FAST_MODE=False before using forward tails as final evidence.')


,forward_paths,forward_median_return,forward_return_05,forward_median_volatility,forward_median_maximum_drawdown,forward_drawdown_95,forward_loss_path_frequency,evidence_tier,ranking_treatment
candidate,,,,,,,,,
Classical strict-warning reference,1000,5.72%,-2.66%,7.51%,7.95%,13.96%,13.5%,FINAL_EVIDENCE,separate_validation_not_scored
Primary unrestricted classical,1000,5.74%,-3.15%,7.86%,8.48%,14.91%,14.1%,FINAL_EVIDENCE,separate_validation_not_scored
Independent Qiskit QAOA,1000,5.70%,-3.19%,7.87%,8.49%,14.93%,14.6%,FINAL_EVIDENCE,separate_validation_not_scored


Final forward-evidence readiness: True


## Step 6F — Visual comparisons

In [55]:
STEP6_OUTPUT = Path(OUTPUT_ROOT) / 'step6_comparison' / DATA_SOURCE / STEP4_COST_SCENARIO
STEP6_FIGURE_DIR = STEP6_OUTPUT / 'figures'
STEP6_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def _short_label(label: str) -> str:
    replacements = {'Primary unrestricted classical': 'Primary classical', 'Independent Qiskit QAOA': 'QAOA', 'Independent exact active-set benchmark': 'Exact active set', 'Classical strict-warning reference': 'Strict warning', 'Classical subset baseline: greedy': 'Greedy', 'Classical subset baseline: local_search': 'Local search', 'Classical-target-recovery exact audit': 'Target recovery', 'Baseline | Current portfolio': 'Current', 'Step 4 | ': 'S4 | '}
    result = label
    for old, new in replacements.items():
        result = result.replace(old, new)
    return result
plot_data = STEP6_RANKING.reset_index()
compliant = plot_data.loc[plot_data['hard_breach_count'].eq(0)]
breached = plot_data.loc[plot_data['hard_breach_count'].gt(0)]
eligible_plot = plot_data.loc[plot_data['eligible_for_selection']].sort_values('selection_rank')
STEP6_PLOT_LABELS = pd.DataFrame({'marker_number': np.arange(1, len(eligible_plot) + 1), 'candidate': eligible_plot['candidate'].to_numpy(), 'short_label': [_short_label(label) for label in eligible_plot['candidate']]}).set_index('marker_number')
print('Numbered labels used in the two clustered Step 6 scatter plots:')
display(STEP6_PLOT_LABELS)
fig = plt.figure(figsize=(12, 7))
plt.scatter(compliant['volatility'], compliant['expected_total_return'], s=45, alpha=0.55, label='Hard-policy compliant')
if not breached.empty:
    plt.scatter(breached['volatility'], breached['expected_total_return'], s=55, marker='x', label='Hard-policy breach')
plt.scatter(eligible_plot['volatility'], eligible_plot['expected_total_return'], s=150, marker='D', label='Unique eligible shortlist')
for marker_number, (_, row) in enumerate(eligible_plot.iterrows(), start=1):
    plt.annotate(str(marker_number), (row['volatility'], row['expected_total_return']), xytext=(6, 6), textcoords='offset points', fontsize=10, fontweight='bold')
plt.xlabel('Annualized volatility')
plt.ylabel('Model-implied expected total return')
plt.title('Step 6: Risk–Return Comparison')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
fig.savefig(STEP6_FIGURE_DIR / 'risk_return_comparison.png', dpi=180, bbox_inches='tight')
plt.show()


Numbered labels used in the two clustered Step 6 scatter plots:


,candidate,short_label
marker_number,,
1,Classical strict-warning reference,Strict warning
2,Primary unrestricted classical,Primary classical
3,Independent Qiskit QAOA,QAOA


In [56]:
fig = plt.figure(figsize=(12, 7))
plt.scatter(compliant['gross_turnover'], compliant['expected_total_return'], s=45, alpha=0.55, label='Hard-policy compliant')
if not breached.empty:
    plt.scatter(breached['gross_turnover'], breached['expected_total_return'], s=55, marker='x', label='Hard-policy breach')
plt.scatter(eligible_plot['gross_turnover'], eligible_plot['expected_total_return'], s=150, marker='D', label='Unique eligible shortlist')
for marker_number, (_, row) in enumerate(eligible_plot.iterrows(), start=1):
    plt.annotate(str(marker_number), (row['gross_turnover'], row['expected_total_return']), xytext=(6, 6), textcoords='offset points', fontsize=10, fontweight='bold')
plt.xlabel('Gross turnover')
plt.ylabel('Model-implied expected total return')
plt.title('Step 6: Return–Implementation Comparison')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
fig.savefig(STEP6_FIGURE_DIR / 'return_turnover_comparison.png', dpi=180, bbox_inches='tight')
plt.show()


In [57]:
warning_plot = STEP6_RANKING[['hard_breach_count', 'warning_breach_count']].sort_values(['hard_breach_count', 'warning_breach_count'], ascending=False)
x_positions = np.arange(len(warning_plot))
fig = plt.figure(figsize=(13, 6))
plt.bar(x_positions, warning_plot['hard_breach_count'], label='Hard breaches')
plt.bar(x_positions, warning_plot['warning_breach_count'], bottom=warning_plot['hard_breach_count'], label='Soft warning breaches')
plt.xticks(x_positions, [_short_label(label) for label in warning_plot.index], rotation=75, ha='right')
plt.ylabel('Breach count')
plt.title('Step 6: Hard-Policy and Soft-Warning Breaches')
plt.grid(True, axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
fig.savefig(STEP6_FIGURE_DIR / 'breach_comparison.png', dpi=180, bbox_inches='tight')
plt.show()


In [58]:
robust_plot = STEP6_SHORTLIST[['robust_mean_score', 'base_decision_score']].sort_values('robust_mean_score', ascending=False)
x_positions = np.arange(len(robust_plot))
fig = plt.figure(figsize=(12, 6))
plt.bar(x_positions - 0.18, robust_plot['robust_mean_score'], width=0.36, label='Mean score across scenarios')
plt.bar(x_positions + 0.18, robust_plot['base_decision_score'], width=0.36, label='Balanced decision score')
plt.xticks(x_positions, [_short_label(label) for label in robust_plot.index], rotation=45, ha='right')
plt.ylabel('Relative shortlist decision score')
plt.title('Step 6: Relative Robust and Balanced Decision Scores')
plt.grid(True, axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
fig.savefig(STEP6_FIGURE_DIR / 'robust_decision_scores.png', dpi=180, bbox_inches='tight')
plt.show()


In [59]:
rank_stability = STEP6_SHORTLIST[['rank_best', 'rank_worst', 'robust_mean_rank']].sort_values('robust_mean_rank')
fig = plt.figure(figsize=(12, 6))
for position, (candidate_name, row) in enumerate(rank_stability.iterrows()):
    plt.plot([row['rank_best'], row['rank_worst']], [position, position], marker='o')
    plt.scatter([row['robust_mean_rank']], [position], marker='x', s=80)
plt.yticks(np.arange(len(rank_stability)), [_short_label(label) for label in rank_stability.index])
plt.xlabel('Rank across decision-weight scenarios')
plt.title('Step 6: Decision-Rank Stability')
plt.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(STEP6_FIGURE_DIR / 'rank_stability.png', dpi=180, bbox_inches='tight')
plt.show()


In [60]:
if not STEP6_CLASS_EXPOSURES.empty:
    exposure_matrix = STEP6_CLASS_EXPOSURES.T
    fig = plt.figure(figsize=(14, max(5, 0.55 * len(exposure_matrix))))
    image = plt.imshow(exposure_matrix.to_numpy(dtype=float), aspect='auto')
    plt.colorbar(image, label='Portfolio weight')
    plt.xticks(np.arange(exposure_matrix.shape[1]), exposure_matrix.columns, rotation=75, ha='right')
    plt.yticks(np.arange(exposure_matrix.shape[0]), [_short_label(label) for label in exposure_matrix.index])
    plt.title('Step 6: Asset-Class Exposure Comparison')
    plt.tight_layout()
    fig.savefig(STEP6_FIGURE_DIR / 'asset_class_exposures.png', dpi=180, bbox_inches='tight')
    plt.show()


## Step 6G — Executive comparison and interpretation

In [61]:
STEP6_NARRATIVES = step6.build_narratives(STEP6_RANKING)
STEP6_EXECUTIVE_TABLE = STEP6_RANKING[['family', 'role', 'expected_total_return', 'volatility', 'worst_scenario_loss', 'in_sample_maximum_drawdown', 'daily_cvar_95', 'gross_turnover', 'total_trading_cost', 'effective_holdings', 'active_trade_count', 'hard_guardrail_status', 'hard_breach_count', 'warning_breach_count', 'guardrail_headroom_10pct', 'method_traceability', 'duplicate_of', 'eligible_for_selection', 'forward_evidence_tier', 'relative_expected_return_score', 'relative_risk_control_score', 'relative_implementation_score', 'relative_guardrail_resilience_score', 'relative_explainability_score', 'relative_robust_mean_score', 'rank_best', 'rank_worst', 'top_1_frequency', 'top_3_frequency', 'selection_rank', 'pareto_comprehensive']].sort_values(['eligible_for_selection', 'selection_rank', 'hard_breach_count'], ascending=[False, True, True])
print('Executive score columns are relative to the current unique eligible shortlist; raw financial metrics remain in their native units.')
display(STEP6_EXECUTIVE_TABLE.style.format({'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'in_sample_maximum_drawdown': '{:.2%}', 'daily_cvar_95': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'effective_holdings': '{:.2f}', 'active_trade_count': '{:.0f}', 'hard_breach_count': '{:.0f}', 'warning_breach_count': '{:.0f}', 'guardrail_headroom_10pct': '{:.2%}', 'method_traceability': '{:.0%}', 'relative_expected_return_score': '{:.1%}', 'relative_risk_control_score': '{:.1%}', 'relative_implementation_score': '{:.1%}', 'relative_guardrail_resilience_score': '{:.1%}', 'relative_explainability_score': '{:.1%}', 'relative_robust_mean_score': '{:.3f}', 'rank_best': '{:.0f}', 'rank_worst': '{:.0f}', 'top_1_frequency': '{:.0%}', 'top_3_frequency': '{:.0%}', 'selection_rank': '{:.0f}'}))
print('Candidate explanations:')
display(STEP6_NARRATIVES.loc[STEP6_EXECUTIVE_TABLE.index])
primary_classical_name = 'Primary unrestricted classical'
qaoa_name = 'Independent Qiskit QAOA'
strict_name = 'Classical strict-warning reference'
if primary_classical_name in STEP6_RANKING.index and qaoa_name in STEP6_RANKING.index:
    classical_row = STEP6_RANKING.loc[primary_classical_name]
    qaoa_row = STEP6_RANKING.loc[qaoa_name]
    print('\nIndependent QAOA relative to the primary classical portfolio:')
    print('Expected-return difference:', f"{qaoa_row['expected_total_return'] - classical_row['expected_total_return']:+.2%}")
    print('Volatility difference:', f"{qaoa_row['volatility'] - classical_row['volatility']:+.2%}")
    print('Worst-scenario difference:', f"{qaoa_row['worst_scenario_loss'] - classical_row['worst_scenario_loss']:+.2%}")
    print('Gross-turnover difference:', f"{qaoa_row['gross_turnover'] - classical_row['gross_turnover']:+.2%}")
    print('Trading-cost difference:', f"{qaoa_row['total_trading_cost'] - classical_row['total_trading_cost']:+.4%}")
if strict_name in STEP6_RANKING.index and primary_classical_name in STEP6_RANKING.index:
    strict_row = STEP6_RANKING.loc[strict_name]
    classical_row = STEP6_RANKING.loc[primary_classical_name]
    print('\nStrict-warning policy relative to the base classical portfolio:')
    print('Expected-return difference:', f"{strict_row['expected_total_return'] - classical_row['expected_total_return']:+.2%}")
    print('Volatility difference:', f"{strict_row['volatility'] - classical_row['volatility']:+.2%}")
    print('Warning-breach difference:', int(strict_row['warning_breach_count'] - classical_row['warning_breach_count']))
    print('Gross-turnover difference:', f"{strict_row['gross_turnover'] - classical_row['gross_turnover']:+.2%}")


Executive score columns are relative to the current unique eligible shortlist; raw financial metrics remain in their native units.


,family,role,expected_total_return,volatility,worst_scenario_loss,in_sample_maximum_drawdown,daily_cvar_95,gross_turnover,total_trading_cost,effective_holdings,active_trade_count,hard_guardrail_status,hard_breach_count,warning_breach_count,guardrail_headroom_10pct,method_traceability,duplicate_of,eligible_for_selection,forward_evidence_tier,relative_expected_return_score,relative_risk_control_score,relative_implementation_score,relative_guardrail_resilience_score,relative_explainability_score,relative_robust_mean_score,rank_best,rank_worst,top_1_frequency,top_3_frequency,selection_rank,pareto_comprehensive
candidate,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Classical strict-warning reference,Step 5 / 5Q,Alternative risk-governance policy,5.48%,7.51%,12.17%,6.53%,0.95%,23.88%,0.0056%,29.35,10,PASS,0,0,10.74%,100%,None,True,FINAL_EVIDENCE,85.7%,100.0%,0.0%,72.9%,65.0%,0.675,1,3,83%,100%,1,True
Primary unrestricted classical,Step 5 / 5Q,Primary continuous benchmark,5.49%,7.86%,12.71%,7.11%,0.99%,13.09%,0.0020%,33.73,6,PASS,0,5,11.24%,100%,None,True,FINAL_EVIDENCE,100.0%,1.7%,83.8%,40.1%,65.3%,0.559,1,2,17%,100%,2,True
Independent Qiskit QAOA,Step 5 / 5Q,Quantum-assisted decision candidate,5.45%,7.87%,12.72%,7.13%,1.00%,11.06%,0.0015%,35.48,4,PASS,0,5,10.44%,65%,None,True,FINAL_EVIDENCE,0.0%,0.0%,100.0%,5.1%,53.2%,0.274,2,3,0%,100%,3,True
Baseline | Current portfolio,Baseline,Incumbent benchmark,5.55%,8.82%,13.83%,8.06%,1.12%,0.00%,0.0000%,40.14,0,PASS,0,5,10.36%,100%,None,False,NOT_APPLICABLE,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan%,nan%,nan,False
Step 4 | 05_trading_costs,Step 4,Investable Step 4 benchmark,5.44%,6.00%,10.48%,5.56%,0.75%,50.00%,0.0198%,20.19,26,PASS,0,2,9.87%,100%,None,False,NOT_APPLICABLE,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan%,nan%,nan,False
Step 4 | 06_scenario_aware,Step 4,Investable Step 4 benchmark,5.38%,5.94%,10.36%,5.24%,0.74%,50.00%,0.0195%,20.70,25,PASS,0,1,9.60%,100%,None,False,NOT_APPLICABLE,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan%,nan%,nan,False
Classical-target-recovery exact audit,Step 5 / 5Q,Diagnostic-only target-recovery audit,5.49%,7.86%,12.71%,7.11%,0.99%,13.09%,0.0020%,33.73,6,PASS,0,5,11.24%,80%,Primary unrestricted classical,False,NOT_APPLICABLE,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan%,nan%,nan,False
Independent exact active-set benchmark,Step 5 / 5Q,Exact reduced-problem benchmark,5.45%,7.87%,12.72%,7.13%,1.00%,11.06%,0.0015%,35.48,4,PASS,0,5,10.44%,95%,Independent Qiskit QAOA,False,NOT_APPLICABLE,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan%,nan%,nan,False
Classical subset baseline: greedy,Step 5 / 5Q,Classical active-set benchmark,5.45%,7.87%,12.72%,7.13%,1.00%,11.06%,0.0015%,35.48,4,PASS,0,5,10.44%,90%,Independent Qiskit QAOA,False,NOT_APPLICABLE,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan%,nan%,nan,False


Candidate explanations:


,return_summary,risk_summary,implementation_summary,governance_summary,explainability_summary
candidate,,,,,
Classical strict-warning reference,Model-implied expected total return 5.48% (2.8...,"Expected volatility 7.51%, worst modeled scena...","Gross turnover 23.88%, estimated trading cost ...",All hard guardrails pass; all scenario warning...,Continuous classical optimization with warning...
Primary unrestricted classical,Model-implied expected total return 5.49% (2.9...,"Expected volatility 7.86%, worst modeled scena...","Gross turnover 13.09%, estimated trading cost ...",All hard guardrails pass; 5 soft scenario warn...,Full continuous classical optimization Method ...
Independent Qiskit QAOA,Model-implied expected total return 5.45% (2.9...,"Expected volatility 7.87%, worst modeled scena...","Gross turnover 11.06%, estimated trading cost ...",All hard guardrails pass; 5 soft scenario warn...,QAOA active-set selection plus continuous refi...
Baseline | Current portfolio,Model-implied expected total return 5.55% (3.1...,"Expected volatility 8.82%, worst modeled scena...","Gross turnover 0.00%, estimated trading cost 0...",All hard guardrails pass; 5 soft scenario warn...,Current strategic allocation Method traceabili...
Step 4 | 05_trading_costs,Model-implied expected total return 5.44% (2.6...,"Expected volatility 6.00%, worst modeled scena...","Gross turnover 50.00%, estimated trading cost ...",All hard guardrails pass; 2 soft scenario warn...,Deterministic classical constraint ladder Meth...
Step 4 | 06_scenario_aware,Model-implied expected total return 5.38% (2.6...,"Expected volatility 5.94%, worst modeled scena...","Gross turnover 50.00%, estimated trading cost ...",All hard guardrails pass; 1 soft scenario warn...,Deterministic classical constraint ladder Meth...
Classical-target-recovery exact audit,Model-implied expected total return 5.49% (2.9...,"Expected volatility 7.86%, worst modeled scena...","Gross turnover 13.09%, estimated trading cost ...",All hard guardrails pass; 5 soft scenario warn...,Exact recovery of an already solved classical ...
Independent exact active-set benchmark,Model-implied expected total return 5.45% (2.9...,"Expected volatility 7.87%, worst modeled scena...","Gross turnover 11.06%, estimated trading cost ...",All hard guardrails pass; 5 soft scenario warn...,Exact active-set enumeration plus continuous r...
Classical subset baseline: greedy,Model-implied expected total return 5.45% (2.9...,"Expected volatility 7.87%, worst modeled scena...","Gross turnover 11.06%, estimated trading cost ...",All hard guardrails pass; 5 soft scenario warn...,Greedy active-set selection plus continuous re...



Independent QAOA relative to the primary classical portfolio:
Expected-return difference: -0.04%
Volatility difference: +0.01%
Worst-scenario difference: +0.00%
Gross-turnover difference: -2.04%
Trading-cost difference: -0.0005%

Strict-warning policy relative to the base classical portfolio:
Expected-return difference: -0.01%
Volatility difference: -0.36%
Warning-breach difference: -5
Gross-turnover difference: +10.79%


## Step 6H — Handoff to Step 7

In [62]:
STEP6_STEP7_HANDOFF = {'context': STEP6_CONTEXT, 'candidate_registry': STEP6_CANDIDATES, 'profile_results': STEP5_PROFILE_RESULTS, 'comparison': STEP6_COMPARISON, 'executive_table': STEP6_EXECUTIVE_TABLE, 'ranking': STEP6_RANKING, 'decision_scenario_scores': STEP6_DECISION_SCENARIO_SCORES, 'score_scope': STEP6_SCORE_SCOPE, 'relative_score_aliases': STEP6_RELATIVE_SCORE_ALIASES, 'score_definitions': STEP6_SCORE_DEFINITIONS, 'forward_validation': STEP6_FORWARD_VALIDATION, 'final_forward_evidence_ready': STEP6_FINAL_FORWARD_EVIDENCE_READY, 'shortlist': STEP6_SHORTLIST, 'primary_recommendation': STEP6_PRIMARY_RECOMMENDATION, 'weights': STEP6_WEIGHT_MATRIX, 'audits': STEP6_AUDITS, 'warning_audits': STEP6_WARNING_AUDITS, 'attributions': STEP6_ATTRIBUTIONS, 'narratives': STEP6_NARRATIVES, 'risk_policy_mode': RISK_POLICY_MODE, 'data_source': DATA_SOURCE}
assert STEP6_SHORTLIST['hard_breach_count'].eq(0).all()
assert STEP6_SHORTLIST['is_unique_portfolio'].all()
assert STEP6_SHORTLIST['decision_eligible_declared'].all()
if 'Classical-target-recovery exact audit' in STEP6_RANKING.index:
    assert not bool(STEP6_RANKING.loc['Classical-target-recovery exact audit', 'eligible_for_selection'])
print('PASS: Step 7 handoff contains only unique, hard-compliant shortlist candidates.')
print('Primary Step 6 recommendation:', STEP6_PRIMARY_RECOMMENDATION)
print('Shortlist:', list(STEP6_SHORTLIST.index))
print('Forward evidence ready for final Step 7 claims:', STEP6_FINAL_FORWARD_EVIDENCE_READY)


PASS: Step 7 handoff contains only unique, hard-compliant shortlist candidates.
Primary Step 6 recommendation: Classical strict-warning reference
Shortlist: ['Classical strict-warning reference', 'Primary unrestricted classical', 'Independent Qiskit QAOA']
Forward evidence ready for final Step 7 claims: True


## Step 6I — Export and download the complete Step 3–6 package

In [63]:
STEP6_OUTPUT.mkdir(parents=True, exist_ok=True)
STEP6_CANDIDATE_REGISTRY.to_csv(STEP6_OUTPUT / 'candidate_registry.csv')
STEP6_COMPARISON.to_csv(STEP6_OUTPUT / 'common_portfolio_comparison.csv')
STEP6_EXECUTIVE_TABLE.to_csv(STEP6_OUTPUT / 'executive_comparison.csv')
STEP6_GUARDRAIL_SUMMARY.to_csv(STEP6_OUTPUT / 'guardrail_summary.csv')
STEP6_BREACH_DETAIL.to_csv(STEP6_OUTPUT / 'hard_breach_detail.csv', index=False)
STEP6_BINDING_GUARDRAILS.to_csv(STEP6_OUTPUT / 'binding_policy_guardrails.csv', index=False)
STEP6_WARNING_BREACH_DETAIL.to_csv(STEP6_OUTPUT / 'soft_warning_breach_detail.csv', index=False)
STEP6_ALL_AUDITS.to_csv(STEP6_OUTPUT / 'all_constraint_audits.csv', index=False)
STEP6_ALL_WARNING_AUDITS.to_csv(STEP6_OUTPUT / 'all_scenario_audits.csv', index=False)
STEP6_WEIGHT_MATRIX.to_csv(STEP6_OUTPUT / 'candidate_weights.csv')
STEP6_EXPLAINABILITY_SUMMARY.to_csv(STEP6_OUTPUT / 'explainability_summary.csv')
STEP6_TOP_TRADES.to_csv(STEP6_OUTPUT / 'top_trades.csv', index=False)
STEP6_TOP_ATTRIBUTIONS.to_csv(STEP6_OUTPUT / 'top_attributions.csv', index=False)
STEP6_CLASS_EXPOSURES.to_csv(STEP6_OUTPUT / 'asset_class_exposures.csv')
STEP6_RANKING.to_csv(STEP6_OUTPUT / 'robust_ranking.csv')
STEP6_DECISION_WEIGHTS.to_csv(STEP6_OUTPUT / 'decision_weight_scenarios.csv')
STEP6_DECISION_SCENARIO_SCORES.to_csv(STEP6_OUTPUT / 'decision_scenario_scores.csv')
STEP6_SCORE_DEFINITIONS.to_csv(STEP6_OUTPUT / 'relative_score_definitions.csv')
STEP6_FORWARD_VALIDATION.to_csv(STEP6_OUTPUT / 'forward_validation.csv')
STEP6_PLOT_LABELS.to_csv(STEP6_OUTPUT / 'plot_label_legend.csv')
STEP6_NARRATIVES.to_csv(STEP6_OUTPUT / 'candidate_narratives.csv')
STEP6_ATTRIBUTION_RECONCILIATION.to_csv(STEP6_OUTPUT / 'attribution_reconciliation.csv')
attribution_root = STEP6_OUTPUT / 'attributions'
attribution_root.mkdir(parents=True, exist_ok=True)

def _safe_filename(label: str) -> str:
    value = re.sub('[^A-Za-z0-9._-]+', '_', label)
    return value.strip('_')
for candidate_name, tables in STEP6_ATTRIBUTIONS.items():
    candidate_dir = attribution_root / _safe_filename(candidate_name)
    candidate_dir.mkdir(parents=True, exist_ok=True)
    for table_name in ['asset', 'asset_class', 'scenario_asset']:
        tables[table_name].to_csv(candidate_dir / f'{table_name}.csv')
step6_metadata = {'method_version': 'step6_comparison_comparison', 'data_source': DATA_SOURCE, 'cost_scenario': STEP4_COST_SCENARIO, 'risk_policy_mode': RISK_POLICY_MODE, 'candidate_count': len(STEP6_CANDIDATES), 'hard_compliant_count': int(STEP6_RANKING['policy_compliant'].sum()), 'unique_eligible_count': int(STEP6_RANKING['eligible_for_selection'].sum()), 'primary_recommendation': STEP6_PRIMARY_RECOMMENDATION, 'shortlist': list(STEP6_SHORTLIST.index), 'synthetic_results_are_not_historical_backtests': DATA_SOURCE == 'synthetic', 'ranking_is_preference_dependent': True, 'score_scope': STEP6_SCORE_SCOPE, 'relative_scores_are_candidate_set_dependent': True, 'forward_results_are_separate_validation_not_scored': True, 'final_forward_evidence_ready': STEP6_FINAL_FORWARD_EVIDENCE_READY, 'target_recovery_is_diagnostic_only': True}
(STEP6_OUTPUT / 'step6_metadata.json').write_text(json.dumps(step6_metadata, indent=2), encoding='utf-8')
required_exports = [STEP6_OUTPUT / 'executive_comparison.csv', STEP6_OUTPUT / 'guardrail_summary.csv', STEP6_OUTPUT / 'robust_ranking.csv', STEP6_OUTPUT / 'candidate_narratives.csv', STEP6_OUTPUT / 'step6_metadata.json', STEP6_OUTPUT / 'relative_score_definitions.csv', STEP6_OUTPUT / 'forward_validation.csv']
missing_exports = [str(path) for path in required_exports if not path.exists()]
if missing_exports:
    raise FileNotFoundError('Step 6 export is incomplete: ' + ', '.join(missing_exports))
archive_base = Path('/content') / f'step3_step4_step5q_step6_finance_comparison_{DATA_SOURCE}_{STEP4_COST_SCENARIO}'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=Path(OUTPUT_ROOT))
print('Created complete Step 3–6 package:', archive_path)
print('Archive size:', f'{Path(archive_path).stat().st_size / 1024 ** 2:.2f} MB')
files.download(archive_path)


Created complete Step 3–6 package: /content/portfolio_pipeline_steps_03_to_06_synthetic_base.zip
Archive size: 2.38 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Step 6 interpretation discipline